# B1 · Transfer learning et modèles pré-entraînés — notebook d'exercices

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cemah2/workbookIA/blob/main/chapitres/b1_transfer/03_notebook.ipynb)

La partie 0 vérifie tes réponses courtes aux quiz, aux rappels et aux exercices papier. Chaque exercice de code : un énoncé, une cellule à compléter (les `...` et les `raise NotImplementedError`), puis une cellule de vérification (`wb.check`, ou des tests). « Tout exécuter » (*Run all*) va jusqu'au bout même si rien n'est rempli : ce qui n'est pas fait affiche ⏳. Les questions « dans tes notes » qui n'ont pas de cellule 📝 se notent dans la section « Notes sur le notebook » de ta copie de `06_mes_reponses.md`. Bloqué 15 minutes ? `04_indices.md`. Le notebook télécharge ResNet-18 et MobileNetV3-small (torchvision) et DINOv2-small (Hugging Face Hub), ≈ 150 Mo, et CIFAR-10 (≈ 145 Mo) s'il n'est pas déjà là ; les embeddings sont calculés une fois puis mis en cache (`data/downloads/embeddings`, hors du dépôt). Tout tourne sur CPU en mode rapide ; 🚀 : en mode complet (`FAST_MODE = False`), 224 pixels et beaucoup plus d'images, sur un GPU de Colab.

> Travaille dans **ta copie** (`mon_travail/b1_transfer/03_notebook.ipynb`, créée par `python tools/start_chapter.py B1`) : ce fichier-ci est mis à jour par Claude. Sur Colab, le badge ouvre cette version du dépôt, qui n'est pas enregistrée : crée puis ouvre ta copie comme l'explique `00_setup/COLAB.md` §2.

| Partie | ID | Titre | Type | ★ | ⏱️ |
|---|---|---|---|---|---|
| 0 | B1.Q1 à B1.Q11, B1.R1 à B1.R3, B1.1 à B1.5 | vérifier tes réponses courtes | 🧠 🔁 ✏️ | ★ à ★★ | |
| A | B1.11 | Trois modèles pré-entraînés en main : ResNet-18, MobileNetV3-small, DINOv2-small | 📦 | ★★ | 20 |
|  | B1.12 | Prédire : un ResNet ImageNet face à des images CIFAR-10 de 32×32 pixels | 🔮 | ★ | 10 |
|  | B1.13 | Extraire des embeddings figés, puis linear probe et k plus proches voisins | 📦 | ★★ | 30 |
|  | B1.14 | Résolution d'entrée : accuracy du linear probe contre temps de calcul (32 à 224 pixels) | 🔬 | ★★★ | 40 |
|  | B1.15 | Lire des projections PCA/UMAP : pixels bruts, ResNet et DINOv2 | 📈 | ★★ | 20 |
|  | B1.16 | Sonder chaque bloc : linear probe couche par couche, CIFAR-10 contre Fashion-MNIST | 🔬 | ★★★ | 45 |
| B | B1.17 | Fine-tuning partiel : dégeler le dernier bloc avec des learning rates par groupe | 📦 | ★★★ | 45 |
|  | B1.18 | Le fine-tuning qui détruit le modèle : quatre erreurs à trouver | 🐛 | ★★ | 25 |
|  | B1.19 | Fine-tuning complet : warmup + cosinus, early stopping, meilleur checkpoint 🚀 | 📦 | ★★★ | 50 |
| C | B1.20 à B1.23 | quand le transfert aide ou échoue : courbes d'apprentissage, domaine lointain, oubli catastrophique (*prochaine étape de génération*) | | | |
| D | B1.24 à B1.28 | modèles de fondation, coût et livraison : zéro-shot avec CLIP, few-shot par prototypes, latence sur CPU, publication du modèle dérivé, et le défi (*prochaine étape de génération*) | | | |

In [ ]:
# ⚙️ SETUP: run this cell first (on Colab or on your computer)
FAST_MODE = True  # True = quick version (CPU, a few minutes); False = full version
SEED = 42

import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/cemah2/workbookIA.git"
DRIVE_DIR = Path("/content/drive/MyDrive/workbookIA")

try:
    import google.colab  # noqa: F401  (this module only exists on Colab)
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import drive

    drive.mount("/content/drive")
    subprocess.run(["git", "config", "--global", "--add", "safe.directory", str(DRIVE_DIR)])
    if not (DRIVE_DIR / ".git").exists():
        subprocess.run(["git", "clone", REPO_URL, str(DRIVE_DIR)], check=True)
    else:
        # Get the new chapters. Your own work only lives in mon_travail/, which Claude never
        # touches, so "rebase + autostash" keeps it safe (even with local commits).
        identity = subprocess.run(["git", "-C", str(DRIVE_DIR), "config", "user.email"],
                                  capture_output=True).returncode == 0
        who = [] if identity else ["-c", "user.name=Workbook", "-c", "user.email=workbook@localhost"]
        pull = subprocess.run(["git", *who, "-C", str(DRIVE_DIR), "pull", "--rebase", "--autostash"],
                              capture_output=True, text=True)
        conflicts = subprocess.run(["git", "-C", str(DRIVE_DIR), "diff", "--name-only", "--diff-filter=U"],
                                   capture_output=True, text=True).stdout.split()
        if conflicts:  # a file changed outside mon_travail/ AND by Claude: git leaves conflict markers in it
            print("⚠️ Tes modifications de " + ", ".join(conflicts) + " (hors de mon_travail/) entrent en conflit "
                  "avec la nouvelle version ; elles sont gardées dans `git stash`. Pour reprendre la version du "
                  "dépôt : git restore --source=HEAD --staged --worktree <fichier> (00_setup/COLAB.md, « Problèmes "
                  "fréquents »).")
        elif pull.returncode != 0:
            print("⚠️ git pull a échoué (voir 00_setup/COLAB.md, « Problèmes fréquents ») :\n" + pull.stderr)
    ROOT = DRIVE_DIR
else:
    ROOT = Path(os.environ.get("WB_ROOT", Path.cwd())).resolve()
    while not (ROOT / "src" / "wb").exists() and ROOT != ROOT.parent:
        ROOT = ROOT.parent
    if not (ROOT / "src" / "wb").exists():
        raise FileNotFoundError("Dépôt introuvable : ouvre ce notebook depuis le dossier du workbook.")

os.environ["WB_ROOT"] = str(ROOT)
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))

import wb  # noqa: E402

cfg = wb.setup(seed=SEED, fast=FAST_MODE)
FAST_MODE = cfg.fast
mylearn = wb.load_mylearn("learner", missing_ok=True, fallback="ref", chapter="B1")  # your own library

## Objectifs et rappel express

- Charger un modèle pré-entraîné avec son prétraitement (torchvision, Hugging Face Hub) et lire sa fiche modèle et sa licence.
- Entraîner un linear probe et un k plus proches voisins sur des embeddings figés, et sonder les blocs d'un backbone.
- Fine-tuner un réseau : dégel progressif, learning rates par groupe, warmup, batchnorm maîtrisée.
- Mesurer le gain du pré-entraînement par des courbes d'apprentissage, et diagnostiquer transfert négatif et oubli catastrophique.
- Classer en zéro-shot et en few-shot avec CLIP et DINOv2, et comparer des backbones sur CPU.

**Rappel express.** Linear probe : $\hat{\mathbf{y}} = \operatorname{softmax}(\mathbf{W}\,\phi(\mathbf{x}) + \mathbf{b})$, $d \times K + K$ paramètres. Learning rates discriminatifs : $\eta_\ell = \eta_{\text{tête}}\,\gamma^{L-\ell}$ ; warmup : $\eta_t = \eta_{\max}\,\min(1, t / t_{\text{warmup}})$. L2-SP : $\Omega = \frac{\lambda}{2}\lVert\theta - \theta_0\rVert^2$. Zéro-shot : $p(c \mid \mathbf{x}) \propto \exp(s \cos(\mathbf{u}_x, \mathbf{v}_c))$. Prototype : $\mathbf{m}_c = \frac{1}{|S_c|}\sum_{\mathbf{x} \in S_c} \phi(\mathbf{x})$.

## Partie 0 · Vérifier tes réponses courtes (quiz, rappels, ✏️ B1.1 à B1.5)

Fais d'abord les quiz, les rappels et les exercices de `02_exercices.md` **sur papier**, dans ta copie de `06_mes_reponses.md`. Reporte ensuite chaque réponse courte ici : **la valeur** que tu as trouvée, pas l'expression Python, sinon tu ne vérifies rien. Un nombre : `42` ou `0.3` (en Python, le séparateur décimal est un **point** ; `0,3` sans guillemets serait un couple de deux nombres) ; un vrai ou faux : `True` ou `False` ; un choix : la lettre seule, entre guillemets (`"B"`) ; plusieurs lettres : entre guillemets (`"AC"`) ; une forme de tenseur : entre guillemets (`"(2, 3)"`) ; un nom de classe : entre guillemets (`"chat"`). Arrondis comme l'énoncé le demande, et seulement à la fin du calcul. Les réponses pas encore remplies affichent ⏳. Les questions « dans ta copie », la dérivation ∂ B1.6, l'estimation 🧮 B1.7, la réflexion (🗣️ B1.8, ⚖️ B1.9, 📄 B1.10) et l'entretien se corrigent avec `05_solutions.md`.

**B1.Q1 — Réutiliser plutôt que repartir de zéro : trois raisons et une limite**

In [ ]:
# B1.Q1: the values you found on paper
answer_B1_Q1a = ...  # a) the letters, e.g. "AC"
answer_B1_Q1b = ...  # b) a whole number
answer_B1_Q1c = ...  # c) True or False
answer_B1_Q1d = ...  # d) the letter of your choice, e.g. "B"
answer_B1_Q1e = ...  # e) True or False

for letter, answer in zip("abcde", [answer_B1_Q1a, answer_B1_Q1b, answer_B1_Q1c, answer_B1_Q1d, answer_B1_Q1e]):
    wb.check(f"B1.Q1{letter}", answer)

**B1.Q2 — Premières couches génériques, dernières couches spécialisées : vrai ou faux**

In [ ]:
# B1.Q2: the values you found on paper
answer_B1_Q2a = ...  # a) True or False
answer_B1_Q2b = ...  # b) True or False
answer_B1_Q2c = ...  # c) True or False
answer_B1_Q2d = ...  # d) True or False
answer_B1_Q2e = ...  # e) True or False

for letter, answer in zip("abcde", [answer_B1_Q2a, answer_B1_Q2b, answer_B1_Q2c, answer_B1_Q2d, answer_B1_Q2e]):
    wb.check(f"B1.Q2{letter}", answer)

**B1.Q3 — Où trouver un modèle pré-entraîné, et quoi lire avant de s'en servir**

In [ ]:
# B1.Q3: the values you found on paper
answer_B1_Q3a = ...  # a) the letter of your choice, e.g. "B"
answer_B1_Q3b = ...  # b) the letters, e.g. "AC"
answer_B1_Q3c = ...  # c) True or False
answer_B1_Q3d = ...  # d) the letter of your choice, e.g. "B"
answer_B1_Q3e = ...  # e) True or False

for letter, answer in zip("abcde", [answer_B1_Q3a, answer_B1_Q3b, answer_B1_Q3c, answer_B1_Q3d, answer_B1_Q3e]):
    wb.check(f"B1.Q3{letter}", answer)

**B1.Q4 — weights.transforms() : pourquoi le prétraitement fait partie du modèle**

In [ ]:
# B1.Q4: the values you found on paper
answer_B1_Q4a = ...  # a) the letter of your choice, e.g. "B"
answer_B1_Q4b = ...  # b) True or False
answer_B1_Q4c = ...  # c) a whole number
answer_B1_Q4d = ...  # d) the letter of your choice, e.g. "B"
answer_B1_Q4e = ...  # e) True or False

for letter, answer in zip("abcde", [answer_B1_Q4a, answer_B1_Q4b, answer_B1_Q4c, answer_B1_Q4d, answer_B1_Q4e]):
    wb.check(f"B1.Q4{letter}", answer)

**B1.Q5 — Extraction de features, linear probe, fine-tuning : qui s'entraîne ?**

In [ ]:
# B1.Q5: the values you found on paper
answer_B1_Q5a = ...  # a) the letter of your choice, e.g. "B"
answer_B1_Q5b = ...  # b) the letter of your choice, e.g. "B"
answer_B1_Q5c = ...  # c) True or False
answer_B1_Q5d = ...  # d) a whole number
answer_B1_Q5e = ...  # e) True or False

for letter, answer in zip("abcde", [answer_B1_Q5a, answer_B1_Q5b, answer_B1_Q5c, answer_B1_Q5d, answer_B1_Q5e]):
    wb.check(f"B1.Q5{letter}", answer)

**B1.Q6 — Pourquoi un learning rate 10 à 100 fois plus petit pour fine-tuner ?**

In [ ]:
# B1.Q6: the values you found on paper
answer_B1_Q6a = ...  # a) the letter of your choice, e.g. "B"
answer_B1_Q6b = ...  # b) True or False
answer_B1_Q6c = ...  # c) the letter of your choice, e.g. "B"
answer_B1_Q6d = ...  # d) True or False
answer_B1_Q6e = ...  # e) the letter of your choice, e.g. "B"

for letter, answer in zip("abcde", [answer_B1_Q6a, answer_B1_Q6b, answer_B1_Q6c, answer_B1_Q6d, answer_B1_Q6e]):
    wb.check(f"B1.Q6{letter}", answer)

**B1.Q7 — Batchnorm pendant le fine-tuning sur de tout petits batches**

In [ ]:
# B1.Q7: the values you found on paper
answer_B1_Q7a = ...  # a) the letter of your choice, e.g. "B"
answer_B1_Q7b = ...  # b) True or False
answer_B1_Q7c = ...  # c) True or False
answer_B1_Q7d = ...  # d) the letter of your choice, e.g. "B"
answer_B1_Q7e = ...  # e) True or False

for letter, answer in zip("abcde", [answer_B1_Q7a, answer_B1_Q7b, answer_B1_Q7c, answer_B1_Q7d, answer_B1_Q7e]):
    wb.check(f"B1.Q7{letter}", answer)

**B1.Q8 — Peu ou beaucoup de données, domaine proche ou lointain : la bonne stratégie**

In [ ]:
# B1.Q8: the values you found on paper
answer_B1_Q8a = ...  # a) the letter of your choice, e.g. "B"
answer_B1_Q8b = ...  # b) the letter of your choice, e.g. "B"
answer_B1_Q8c = ...  # c) the letter of your choice, e.g. "B"
answer_B1_Q8d = ...  # d) True or False
answer_B1_Q8e = ...  # e) True or False

for letter, answer in zip("abcde", [answer_B1_Q8a, answer_B1_Q8b, answer_B1_Q8c, answer_B1_Q8d, answer_B1_Q8e]):
    wb.check(f"B1.Q8{letter}", answer)

**B1.Q9 — Transfert négatif et oubli catastrophique : reconnaître les symptômes**

In [ ]:
# B1.Q9: the values you found on paper
answer_B1_Q9a = ...  # a) the letter of your choice, e.g. "B"
answer_B1_Q9b = ...  # b) the letter of your choice, e.g. "B"
answer_B1_Q9c = ...  # c) True or False
answer_B1_Q9d = ...  # d) True or False
answer_B1_Q9e = ...  # e) the letter of your choice, e.g. "B"

for letter, answer in zip("abcde", [answer_B1_Q9a, answer_B1_Q9b, answer_B1_Q9c, answer_B1_Q9d, answer_B1_Q9e]):
    wb.check(f"B1.Q9{letter}", answer)

**B1.Q10 — Classer sans exemple d'entraînement : le zéro-shot de CLIP**

In [ ]:
# B1.Q10: the values you found on paper
answer_B1_Q10a = ...  # a) the letter of your choice, e.g. "B"
answer_B1_Q10b = ...  # b) True or False
answer_B1_Q10c = ...  # c) True or False
answer_B1_Q10d = ...  # d) the letter of your choice, e.g. "B"
answer_B1_Q10e = ...  # e) True or False

for letter, answer in zip("abcde", [answer_B1_Q10a, answer_B1_Q10b, answer_B1_Q10c, answer_B1_Q10d, answer_B1_Q10e]):
    wb.check(f"B1.Q10{letter}", answer)

**B1.Q11 — Auto-supervisé (DINOv2) ou supervisé sur ImageNet : quels embeddings ?**

In [ ]:
# B1.Q11: the values you found on paper
answer_B1_Q11a = ...  # a) the letter of your choice, e.g. "B"
answer_B1_Q11b = ...  # b) True or False
answer_B1_Q11c = ...  # c) the letter of your choice, e.g. "B"
answer_B1_Q11d = ...  # d) True or False
answer_B1_Q11e = ...  # e) a whole number

for letter, answer in zip("abcde", [answer_B1_Q11a, answer_B1_Q11b, answer_B1_Q11c, answer_B1_Q11d, answer_B1_Q11e]):
    wb.check(f"B1.Q11{letter}", answer)

**B1.R1 — Ch. 29 : licence d'un dataset et décalage de distribution**

In [ ]:
# B1.R1: the values you found on paper
answer_B1_R1a = ...  # a) True or False
answer_B1_R1b = ...  # b) the letter of your choice, e.g. "B"
answer_B1_R1c = ...  # c) the letter of your choice, e.g. "B"
answer_B1_R1d = ...  # d) True or False
answer_B1_R1e = ...  # e) the letter of your choice, e.g. "B"

for letter, answer in zip("abcde", [answer_B1_R1a, answer_B1_R1b, answer_B1_R1c, answer_B1_R1d, answer_B1_R1e]):
    wb.check(f"B1.R1{letter}", answer)

**B1.R2 — Ch. 23 : geler un réseau et remplacer sa tête en PyTorch**

In [ ]:
# B1.R2: the values you found on paper
answer_B1_R2a = ...  # a) the letter of your choice, e.g. "B"
answer_B1_R2b = ...  # b) True or False
answer_B1_R2c = ...  # c) a whole number
answer_B1_R2d = ...  # d) True or False
answer_B1_R2e = ...  # e) the letter of your choice, e.g. "B"

for letter, answer in zip("abcde", [answer_B1_R2a, answer_B1_R2b, answer_B1_R2c, answer_B1_R2d, answer_B1_R2e]):
    wb.check(f"B1.R2{letter}", answer)

**B1.R3 — Ch. 21 : ce que détectent les filtres des premières et des dernières couches**

In [ ]:
# B1.R3: the values you found on paper
answer_B1_R3a = ...  # a) the letter of your choice, e.g. "B"
answer_B1_R3b = ...  # b) a whole number
answer_B1_R3c = ...  # c) True or False
answer_B1_R3d = ...  # d) True or False
answer_B1_R3e = ...  # e) the letter of your choice, e.g. "B"

for letter, answer in zip("abcde", [answer_B1_R3a, answer_B1_R3b, answer_B1_R3c, answer_B1_R3d, answer_B1_R3e]):
    wb.check(f"B1.R3{letter}", answer)

**Ex B1.1 — Nouvelle tête : formes des tenseurs de l'image 32×32 redimensionnée jusqu'aux logits**

In [ ]:
# Ex B1.1: the values you found on paper
answer_B1_1a = ...  # a) a shape as a string, e.g. "(2, 3)"
answer_B1_1b = ...  # b) a shape as a string, e.g. "(2, 3)"
answer_B1_1c = ...  # c) a shape as a string, e.g. "(2, 3)"
answer_B1_1d = ...  # d) a shape as a string, e.g. "(2, 3)"
answer_B1_1e = ...  # e) a shape as a string, e.g. "(2, 3)"
answer_B1_1f = ...  # f) a whole number
answer_B1_1g = ...  # g) a whole number
answer_B1_1h = ...  # h) True or False

for letter, answer in zip("abcdefgh", [answer_B1_1a, answer_B1_1b, answer_B1_1c, answer_B1_1d, answer_B1_1e, answer_B1_1f, answer_B1_1g, answer_B1_1h]):
    wb.check(f"B1.1{letter}", answer)

**Ex B1.2 — k plus proches voisins cosinus sur six embeddings donnés**

In [ ]:
# Ex B1.2: the values you found on paper
answer_B1_2a = ...  # a) a number, 3 decimals
answer_B1_2b = ...  # b) a vector number, from 0 to 5
answer_B1_2c = ...  # c) a class name as a string, e.g. "chat"
answer_B1_2d = ...  # d) a class name as a string, e.g. "chat"
answer_B1_2e = ...  # e) a vector number, from 0 to 5
answer_B1_2f = ...  # f) a number, 3 decimals
answer_B1_2g = ...  # g) True or False

for letter, answer in zip("abcdefg", [answer_B1_2a, answer_B1_2b, answer_B1_2c, answer_B1_2d, answer_B1_2e, answer_B1_2f, answer_B1_2g]):
    wb.check(f"B1.2{letter}", answer)

**Ex B1.3 — Learning rates discriminatifs et warmup : le learning rate de chaque groupe, pas à pas**

In [ ]:
# Ex B1.3: the values you found on paper
answer_B1_3a = ...  # a) a number, 1 decimal
answer_B1_3b = ...  # b) a whole number
answer_B1_3c = ...  # c) a whole number
answer_B1_3d = ...  # d) a number, 2 decimals
answer_B1_3e = ...  # e) a number, 2 decimals
answer_B1_3f = ...  # f) a number, 2 decimals
answer_B1_3g = ...  # g) True or False
answer_B1_3h = ...  # h) True or False

for letter, answer in zip("abcdefgh", [answer_B1_3a, answer_B1_3b, answer_B1_3c, answer_B1_3d, answer_B1_3e, answer_B1_3f, answer_B1_3g, answer_B1_3h]):
    wb.check(f"B1.3{letter}", answer)

**Ex B1.4 — Compter les paramètres entraînables : tête seule, dernier bloc, réseau entier**

In [ ]:
# Ex B1.4: the values you found on paper
answer_B1_4a = ...  # a) a whole number
answer_B1_4b = ...  # b) a whole number
answer_B1_4c = ...  # c) a whole number
answer_B1_4d = ...  # d) a whole number
answer_B1_4e = ...  # e) a whole number
answer_B1_4f = ...  # f) a number, 1 decimal
answer_B1_4g = ...  # g) a whole number
answer_B1_4h = ...  # h) True or False

for letter, answer in zip("abcdefgh", [answer_B1_4a, answer_B1_4b, answer_B1_4c, answer_B1_4d, answer_B1_4e, answer_B1_4f, answer_B1_4g, answer_B1_4h]):
    wb.check(f"B1.4{letter}", answer)

**Ex B1.5 — Zéro-shot CLIP à la main : cosinus, logit_scale et softmax**

In [ ]:
# Ex B1.5: the values you found on paper
answer_B1_5a = ...  # a) a number, 3 decimals
answer_B1_5b = ...  # b) a number, 3 decimals
answer_B1_5c = ...  # c) a number, 3 decimals
answer_B1_5d = ...  # d) a number, 4 decimals
answer_B1_5e = ...  # e) a number, 2 decimals
answer_B1_5f = ...  # f) True or False

for letter, answer in zip("abcdef", [answer_B1_5a, answer_B1_5b, answer_B1_5c, answer_B1_5d, answer_B1_5e, answer_B1_5f]):
    wb.check(f"B1.5{letter}", answer)

## Outils du notebook

La cellule suivante importe PyTorch, charge les sous-ensembles de CIFAR-10 (`X_TRAIN`, `X_VAL`, `X_TEST`), le ResNet-18 gelé de référence `RESNET` et définit les petits outils des parties A et B : `nested`, `reference_embeddings`, `reference_probe`, `pct`… Lis leurs docstrings au besoin. Le premier lancement télécharge CIFAR-10 et les poids de ResNet-18.

In [ ]:
# Tools for the notebook exercises (parts A and B)
import copy
import math
import time
import warnings

import matplotlib.pyplot as plt
import numpy as np
from wb.nbtools import StopCheck, checked, filled, fr, returned, same, verdict

try:
    import torch
    import torch.nn as nn
    import torch.nn.functional as F
except ImportError:                       # a computer without PyTorch (Mac Intel): this chapter is done on Colab
    torch = None
    print("⏳ PyTorch absent sur cet ordinateur : ce chapitre se fait sur Colab (00_setup/INSTALL_LOCAL.md).")

warnings.filterwarnings("ignore", message=".*IProgress not found.*")    # tqdm without ipywidgets (transformers)
from wb import transfer
from wb.transfer import DINOV2_NAME, DINOV2_REVISION

DEVICE = cfg.device                       # "cuda" on a Colab GPU, else "cpu"
SIZE = 96 if FAST_MODE else 224           # the side of the images given to ResNet-18 (CIFAR-10 images are 32 x 32)
DINO_SIZE = 140 if FAST_MODE else 224     # the side of the images given to DINOv2 (a multiple of its 14-pixel patches)
PER_CLASS = 200 if FAST_MODE else 1000    # training images per class (nested, stratified subsets of CIFAR-10)
TEST_PER_CLASS = 200 if FAST_MODE else 1000   # test images per class (1 000 = the whole test split)
VAL_PER_CLASS = 50 if FAST_MODE else 100      # validation images per class, never in a training subset
CLASSES = wb.datasets.CIFAR10_CLASSES
X_TRAIN, Y_TRAIN = wb.datasets.load_cifar10_subset(PER_CLASS)          # uint8 (N, 32, 32, 3), int64 labels
X_TEST, Y_TEST = wb.datasets.load_cifar10_subset(TEST_PER_CLASS, "test")
X_VAL, Y_VAL = wb.datasets.load_cifar10_subset(VAL_PER_CLASS, "val")
# The frozen REFERENCE backbone of the checks and of the given experiments (wb.transfer), not the one you write:
# RESNET.model(x) gives the 512 values of the embedding, RESNET.full is the whole network (ImageNet head included)
RESNET = transfer.load_backbone("resnet18")


def pct(x, digits=1):
    """A share written the French way: 0.734 -> '73,4 %'."""
    return f"{100 * x:.{digits}f} %".replace(".", ",")


def nested(per_class, part="train"):
    """The positions, in X_TRAIN (part="train") or X_TEST ("test"), of the smaller nested subset of `per_class`
    images per class (the subsets are nested: the 30 per class are among the 200)."""
    big = wb.datasets.cifar10_subset_indices(PER_CLASS if part == "train" else TEST_PER_CLASS, part)
    return np.searchsorted(big, wb.datasets.cifar10_subset_indices(per_class, part))


def reference_embeddings(images, size, subset, block=None):
    """The embeddings of the reference ResNet-18, computed once and then read from the cache (data/downloads)."""
    return transfer.cached_embeddings(RESNET, images, size=size, block=block, subset=subset, verbose=False)


def reference_probe(E_train, y_train, E_test, y_test, C=1.0):
    """Test accuracy of the reference linear probe (StandardScaler, then LogisticRegression)."""
    from sklearn.linear_model import LogisticRegression
    from sklearn.pipeline import make_pipeline
    from sklearn.preprocessing import StandardScaler
    probe = make_pipeline(StandardScaler(), LogisticRegression(C=C, max_iter=2000))
    return float(probe.fit(E_train, y_train).score(E_test, y_test))


def n_params(model, trainable_only=False):
    return sum(p.numel() for p in model.parameters() if p.requires_grad or not trainable_only)


def need(ex_id, *names):
    """Stop a check cell with ⏳ while a function of an earlier exercise is not written yet."""
    for name in names:
        fn = globals().get(name)
        if fn is None:
            print(f"⏳ Ex {ex_id} : définis d'abord {name}.")
            raise StopCheck


torch.manual_seed(0)
print(f"CIFAR-10 : {len(X_TRAIN)} images d'entraînement, {len(X_VAL)} de validation, {len(X_TEST)} de test ; "
      f"ResNet-18 à {SIZE} pixels, DINOv2 à {DINO_SIZE} pixels ; calcul sur {DEVICE}.")

## Partie A · Charger et sonder : modèles pré-entraînés, embeddings figés, linear probe

Trois modèles pré-entraînés et leur prétraitement, puis l'extraction de features : un backbone gelé transforme chaque image de CIFAR-10 en un embedding, calculé une fois, sur lequel on entraîne un linear probe ou un k plus proches voisins. Les cellules de vérification comparent ton code à un backbone de référence, `RESNET` (dans les outils : ResNet-18 gelé, `wb.transfer`), dont les embeddings sont calculés une seule fois puis rangés dans un cache sur le disque (`data/downloads/embeddings`, hors du dépôt) : relancer une cellule ne refait pas le calcul. Les images viennent de sous-ensembles **emboîtés et stratifiés** de CIFAR-10, à graine fixée : `X_TRAIN` (`PER_CLASS` images par classe), `X_VAL` (des images d'entraînement qui ne sont dans aucun sous-ensemble d'entraînement), `X_TEST` ; `nested(k)` donne la position, dans `X_TRAIN`, des $k$ images par classe d'un sous-ensemble plus petit. En mode rapide, ResNet-18 reçoit des images agrandies à `SIZE` = 96 pixels (224 en mode complet).

### Ex B1.11 — Trois modèles pré-entraînés en main : ResNet-18, MobileNetV3-small, DINOv2-small 📦 ★★ ⏱️ 20 min
**Objectif :** charger trois modèles pré-entraînés avec leur prétraitement, et lire sur les objets ce que la documentation ne dit pas.  
**Prérequis :** ch. 23 (modèles de torchvision) · fiche §201.2 (encadrés 🧮 sur `weights.transforms()` et l'API transformers), §201.5 (la nouvelle tête) · **Fil rouge :** CIFAR-10 · **Parcours :** R, M, C

**Contrat** de `load_backbones_11()` : un dictionnaire de trois couples, chacun en mode `eval()` :
- `"resnet18"` et `"mobilenet_v3_small"` : `(modèle, prétraitement)`, le modèle de torchvision avec ses poids `IMAGENET1K_V1` (tête ImageNet comprise), le prétraitement `weights.transforms()` de ces poids ;
- `"dinov2_small"` : `(modèle, processeur)`, le backbone `AutoModel` du dépôt `DINOV2_NAME` (`facebook/dinov2-small`) à la révision `DINOV2_REVISION` (un commit fixé, dans les outils), et son `AutoImageProcessor` à la même révision.

Le premier appel télécharge les poids (≈ 45, 10 et 90 Mo), les suivants les relisent dans le cache.

Ensuite, **lis sur les objets** (dans une cellule à toi : `print(model)`, `print(preprocess)`, `model.config`, une passe avant) et reporte les **valeurs** :
a) la taille du vecteur à l'**entrée** de `mobilenet.classifier` ;
b) la taille du vecteur **entre** ses deux couches denses ;
c) `hidden_size` de DINOv2-small ;
d) `True` ou `False` : `pooler_output` est égal à `last_hidden_state[:, 0]` (essaie sur deux images de `X_TEST`, passées par le processeur) ;
e) `resize_size` du prétraitement de ResNet-18 ;
f) les lettres des modèles dont le prétraitement interpole en **bicubique** : (A) ResNet-18, (B) MobileNetV3-small, (C) DINOv2-small.

La cellule de vérification contrôle chaque valeur, puis compare tes trois modèles aux modèles de référence (poids, mode, prétraitement).

Dans tes notes : pour MobileNetV3-small, lequel des deux vecteurs de a) et b) prendrais-tu comme embedding, et pourquoi (pense à ce que la première couche dense a appris) ? Que contient la fiche modèle de `facebook/dinov2-small` sur le Hub (licence, données, usages) ?

In [ ]:
def load_backbones_11():
    """The three pre-trained models of the chapter, each with its preprocessing, in eval() mode.

    Returns {"resnet18": (model, preprocess), "mobilenet_v3_small": (model, preprocess),
             "dinov2_small": (model, processor)}:
    - the two torchvision models with their IMAGENET1K_V1 weights (ImageNet head kept), preprocess = weights.transforms();
    - DINOv2-small from the Hugging Face Hub (DINOV2_NAME, revision DINOV2_REVISION): AutoModel and AutoImageProcessor."""
    raise NotImplementedError("load_backbones_11")


# What you read on the objects, in a cell of your own (the values, e.g. 42 or "AB")
mobilenet_in_11 = ...     # a) the size of the vector at the ENTRANCE of mobilenet.classifier
mobilenet_mid_11 = ...    # b) the size of the vector BETWEEN the two dense layers of mobilenet.classifier
dino_d_11 = ...           # c) hidden_size of DINOv2-small
pooler_is_cls_11 = ...    # d) True or False: pooler_output equals last_hidden_state[:, 0] (try two images)
resize_11 = ...           # e) resize_size of the preprocessing of ResNet-18 (a whole number)
bicubic_11 = ...          # f) the letters of the models whose preprocessing interpolates in BICUBIC:
                          #    (A) ResNet-18, (B) MobileNetV3-small, (C) DINOv2-small, e.g. "AB"

In [ ]:
# a) to f): the values you read (each one is checked on its own)
for id_11, value_11 in (("B1.11a", mobilenet_in_11), ("B1.11b", mobilenet_mid_11), ("B1.11c", dino_d_11),
                        ("B1.11d", pooler_is_cls_11), ("B1.11e", resize_11), ("B1.11f", bicubic_11)):
    with wb.attempt(id_11):
        wb.check(id_11, value_11)

# load_backbones_11: compared with the reference models of wb.transfer
with checked("B1.11"):
    from torchvision.models import MobileNet_V3_Small_Weights, ResNet18_Weights
    loaded_11 = load_backbones_11()
    if not returned("B1.11", "load_backbones_11", loaded_11):
        raise StopCheck
    keys_11 = {"resnet18", "mobilenet_v3_small", "dinov2_small"}
    if not isinstance(loaded_11, dict) or set(loaded_11) != keys_11:
        print(f"❌ Ex B1.11 : attendu un dictionnaire aux clés {sorted(keys_11)}, reçu {type(loaded_11).__name__} "
              f"{sorted(loaded_11) if isinstance(loaded_11, dict) else ''}.")
        raise StopCheck
    for name_11 in sorted(keys_11):
        if not (isinstance(loaded_11[name_11], (tuple, list)) and len(loaded_11[name_11]) == 2):
            print(f"❌ Ex B1.11 : loaded['{name_11}'] doit être un couple (modèle, prétraitement).")
            raise StopCheck
    for name_11, weights_11 in (("resnet18", ResNet18_Weights.IMAGENET1K_V1),
                                ("mobilenet_v3_small", MobileNet_V3_Small_Weights.IMAGENET1K_V1)):
        model_11, prep_11 = loaded_11[name_11]
        ref_11 = transfer.load_backbone(name_11).full.state_dict()
        mine_11 = model_11.state_dict()
        same_11 = set(mine_11) == set(ref_11) and all(torch.equal(mine_11[k], ref_11[k]) for k in ref_11)
        verdict("B1.11", same_11, f"{name_11} : les poids IMAGENET1K_V1, tête ImageNet comprise.",
                f"{name_11} : ce ne sont pas les poids IMAGENET1K_V1 avec leur tête ImageNet (autres poids, tête "
                "remplacée, ou poids modifiés ?).")
        verdict("B1.11", not model_11.training, f"{name_11} est en mode eval().",
                f"{name_11} est en mode train() : un modèle torchvision se crée en mode entraînement.")
        verdict("B1.11", str(prep_11) == str(weights_11.transforms()), f"{name_11} : le prétraitement de ses poids.",
                f"{name_11} : attendu le prétraitement de ses poids, reçu {str(prep_11)[:120]!r}.")
    dino_ref_11 = transfer.try_backbone("dinov2_small", "B1.11")
    if dino_ref_11 is not None:
        model_11, processor_11 = loaded_11["dinov2_small"]
        mine_11, ref_11 = model_11.state_dict(), dino_ref_11.full.state_dict()
        same_11 = set(mine_11) == set(ref_11) and all(torch.equal(mine_11[k], ref_11[k]) for k in ref_11)
        verdict("B1.11", same_11 and getattr(getattr(model_11, "config", None), "hidden_size", None) == 384,
                "dinov2_small : le backbone de facebook/dinov2-small, sans tête.",
                "dinov2_small : ce n'est pas le backbone de facebook/dinov2-small chargé par AutoModel.")
        verdict("B1.11", not model_11.training, "dinov2_small est en mode eval().", "dinov2_small est en mode train().")
        crop_11 = getattr(processor_11, "crop_size", None)
        verdict("B1.11", getattr(processor_11, "resample", None) == 3 and getattr(crop_11, "height", None) == 224
                and tuple(getattr(processor_11, "image_mean", ())) == wb.datasets.IMAGENET_MEAN,
                "dinov2_small : son processeur d'images (recadrage 224, interpolation bicubique, statistiques d'ImageNet).",
                f"dinov2_small : attendu le processeur d'images du dépôt (AutoImageProcessor), reçu {type(processor_11).__name__}.")

### Ex B1.12 — Prédire : un ResNet ImageNet face à des images CIFAR-10 de 32×32 pixels 🔮 ★ ⏱️ 10 min
**Objectif :** prédire, en chiffres, ce que garde un ResNet entraîné sur des images de 224 pixels quand on lui donne des images de 32 pixels.  
**Prérequis :** Ex B1.11 · ✏️ B1.1 f) · fiche §201.5 (la résolution) · **Fil rouge :** CIFAR-10 · **Parcours :** C

ResNet-18 a appris sur des images de 224 × 224 ; celles de CIFAR-10 mesurent 32 × 32. Le ✏️ B1.1 f) t'a fait suivre la forme : à 32 pixels, la carte de `layer4` n'a plus qu'**une** position. La cellule d'expérience donne les images **telles quelles** (32 × 32, normalisées) au ResNet-18 de référence, puis agrandies à `SIZE` pixels, et mesure deux choses :
- l'accuracy d'un linear probe entraîné sur les embeddings de `X_TRAIN`, mesurée sur `X_TEST` ;
- les classes **ImageNet** que prédit la tête d'origine du réseau (ses 1 000 sorties) pour les images de test.

**Avant d'exécuter**, écris tes deux prédictions, **pour les images de 32 pixels** :
a) l'accuracy du linear probe : `"A"` moins de 30 %, `"B"` de 30 à 70 %, `"C"` plus de 70 % ;
b) la part des images de test dont la classe ImageNet prédite est l'une des **5 classes les plus fréquentes** parmi les prédictions : `"A"` moins de 30 %, `"B"` de 30 à 70 %, `"C"` plus de 70 %.

La cellule d'expérience n'affiche rien tant que tes deux prédictions ne sont pas écrites (≈ 30 s la première fois : les embeddings sont calculés puis mis en cache).

📝 **Mon hypothèse** (à écrire **avant** d'exécuter quoi que ce soit) : …

In [ ]:
probe32_12 = ...   # a) the linear probe accuracy on the embeddings of the images at 32 x 32 (NOT resized):
                   #    "A" below 30 %, "B" from 30 to 70 %, "C" above 70 %
top5_12 = ...      # b) at 32 x 32, the share of the test images whose predicted ImageNet class is one of the 5 most
                   #    frequent predicted classes: "A" below 30 %, "B" from 30 to 70 %, "C" above 70 %

In [ ]:
if any(answer is ... or answer is None for answer in [probe32_12, top5_12]):
    print("⏳ Ex B1.12 : écris d'abord tes deux prédictions dans la cellule précédente.")
else:
    with checked("B1.12"):
        rows_12 = []
        for size_12 in (32, SIZE):
            E_train_12 = reference_embeddings(X_TRAIN, size_12, "train")
            E_test_12 = reference_embeddings(X_TEST, size_12, "test")
            acc_12 = reference_probe(E_train_12, Y_TRAIN, E_test_12, Y_TEST)
            with torch.inference_mode():
                imagenet_12 = RESNET.full.fc(torch.from_numpy(E_test_12)).argmax(1).numpy()   # the ImageNet head
            names_12, counts_12 = np.unique(imagenet_12, return_counts=True)
            top_12 = np.argsort(-counts_12, kind="stable")[:5]
            share_12 = counts_12[top_12].sum() / len(imagenet_12)
            print(f"{size_12:>3} pixels : linear probe {pct(acc_12)} ; {len(names_12)} classes ImageNet différentes prédites "
                  f"pour {len(imagenet_12)} images ; les 5 plus fréquentes ({', '.join(RESNET.categories[names_12[k]] for k in top_12)}) "
                  f"couvrent {pct(share_12)} des images.")
        print("(le hasard : 10 % ; linear probe entraîné sur", len(X_TRAIN), "images, mesuré sur", len(X_TEST), ")")
        wb.check("B1.12a", probe32_12)
        wb.check("B1.12b", top5_12)

Dans tes notes : tes prédictions étaient-elles justes ? Compare les deux lignes (32 pixels et `SIZE` pixels). Une seule position dans `layer4` : que reste-t-il pour le linear probe ? Les classes ImageNet prédites ressemblent-elles aux classes de CIFAR-10 ?

### Ex B1.13 — Extraire des embeddings figés, puis linear probe et k plus proches voisins 📦 ★★ ⏱️ 30 min
**Objectif :** extraire les embeddings figés d'un backbone pré-entraîné, puis les classer par un linear probe et par les k plus proches voisins.  
**Prérequis :** Ex B1.11 · ch. 15 (Pipeline) · ch. 13 (k-NN) · ✏️ B1.2 · fiche §201.3 · **Fil rouge :** CIFAR-10 · **Parcours :** R, M, C

**Contrat**
- `make_backbone_13()` : ton ResNet-18 de B1.11 (poids `IMAGENET1K_V1`), sa tête `fc` remplacée par `nn.Identity()`, en mode `eval()` : sa sortie est l'embedding $\phi(\mathbf{x})$, de taille 512 ;
- `embed_13(backbone, images, size, batch_size=64)` : pour des images `uint8` de forme `(N, 32, 32, 3)`, laissées intactes, un tableau NumPy `float32` de forme `(N, d)`, calculé par mini-batches de `batch_size`, sans gradient. Chaque image passe en `float` dans $[0, 1]$, canaux en premier, agrandie à `size` × `size` (interpolation bilinéaire, sans recadrage), puis normalisée par les moyennes et écarts-types d'ImageNet (`wb.datasets.IMAGENET_MEAN`, `IMAGENET_STD`, un par canal) ;
- `probe_13(E_train, y_train, E_test, y_test, C=1.0)` : l'accuracy de test (un `float`) d'un linear probe, une `StandardScaler` puis une `LogisticRegression(C=C, max_iter=2000)` ;
- `knn_13(E_train, y_train, E_test, y_test, k=10)` : l'accuracy de test des `k` plus proches voisins pour la similarité cosinus.

La cellule de vérification compare tes embeddings de 16 images à ceux de `RESNET`, vérifie qu'ils ne dépendent pas de la taille des mini-batches, puis lance ton probe et ton k-NN sur les embeddings en cache de `X_TRAIN` et de `X_TEST`, avec plusieurs réglages. La cellule suivante choisit `C` et `k` sur la **validation** et mesure le couple choisi une fois sur le test.

Dans tes notes : le linear probe et le k-NN se valent-ils ? Lequel préférerais-tu si des classes nouvelles arrivaient chaque semaine ? Pourquoi choisir `C` et `k` sur `X_VAL` et pas sur `X_TEST` ?

In [ ]:
def make_backbone_13():
    """Your ResNet-18 of B1.11 turned into a backbone: IMAGENET1K_V1 weights, fc replaced by nn.Identity(), eval()."""
    raise NotImplementedError("make_backbone_13")


def embed_13(backbone, images, size, batch_size=64):
    """Frozen embeddings of CIFAR-10 images: a float32 NumPy array (N, d).

    backbone: a network whose output is the embedding, in eval() mode; images: uint8 (N, 32, 32, 3), left unchanged;
    size: the side of the resized images. By mini-batches of batch_size, without gradient: pixels in [0, 1],
    channels first, resized to size x size (bilinear), normalised with the ImageNet statistics."""
    raise NotImplementedError("embed_13")


def probe_13(E_train, y_train, E_test, y_test, C=1.0):
    """Test accuracy (a float) of a linear probe: StandardScaler, then LogisticRegression(C=C, max_iter=2000)."""
    raise NotImplementedError("probe_13")


def knn_13(E_train, y_train, E_test, y_test, k=10):
    """Test accuracy (a float) of the k nearest neighbours for the cosine similarity."""
    raise NotImplementedError("knn_13")

In [ ]:
with checked("B1.13"):
    backbone_13 = make_backbone_13()
    if not returned("B1.13", "make_backbone_13", backbone_13):
        raise StopCheck
    verdict("B1.13", not backbone_13.training, "le backbone est en mode eval().",
            "le backbone est en mode train() : ses batchnorms normaliseraient avec les statistiques de chaque mini-batch.")
    images_13 = X_TEST[:16].copy()
    E_13 = embed_13(backbone_13, images_13, SIZE)
    if not returned("B1.13", "embed_13", E_13):
        raise StopCheck
    verdict("B1.13", np.array_equal(images_13, X_TEST[:16]), "embed_13 laisse les images intactes.",
            "embed_13 a modifié les images reçues : travaille sur une copie convertie.")
    if not isinstance(E_13, np.ndarray):
        print(f"❌ Ex B1.13 : embed_13 doit rendre un tableau NumPy, reçu {type(E_13).__name__}"
              + (" (convertis avec .numpy())." if torch.is_tensor(E_13) else "."))
        raise StopCheck
    want_13 = RESNET.embed(X_TEST[:16], SIZE)
    if E_13.shape != want_13.shape:
        print(f"❌ Ex B1.13 : embed_13(backbone, X_TEST[:16], SIZE) : attendu la forme {want_13.shape}, reçu {E_13.shape}.")
        raise StopCheck
    if not np.allclose(E_13, want_13, rtol=1e-3, atol=1e-3):
        variants_13 = {   # what a frequent slip gives, to say where to look
            "sans la normalisation d'ImageNet": transfer.to_input(X_TEST[:16], SIZE, (0, 0, 0), (1, 1, 1)),
            "avec une interpolation bicubique": transfer.to_input(X_TEST[:16], SIZE, resample="bicubic"),
            "sans agrandir les images": transfer.to_input(X_TEST[:16], None),
        }
        with torch.inference_mode():
            hit_13 = [k for k, x in variants_13.items() if np.allclose(E_13, RESNET.model(x).numpy(), atol=1e-3)]
        print("❌ Ex B1.13 : embed_13 ne rend pas les embeddings du contrat"
              + (f" : tes valeurs sont celles d'images préparées {hit_13[0]}." if hit_13 else
                 " : vérifie l'ordre des axes, la division par 255, l'agrandissement à size × size et la normalisation."))
        raise StopCheck
    verdict("B1.13", E_13.dtype == np.float32, "embed_13 rend les embeddings attendus, en float32.",
            f"embed_13 rend du {E_13.dtype} au lieu de float32.")
    E_small_13 = embed_13(backbone_13, X_TEST[:16].copy(), SIZE, batch_size=5)
    verdict("B1.13", isinstance(E_small_13, np.ndarray) and np.allclose(E_small_13, E_13, atol=1e-4),
            "l'embedding d'une image ne dépend pas des images de son mini-batch.",
            "avec batch_size=5, les embeddings changent : le backbone est-il en mode eval(), toutes les images "
            "sont-elles rendues ?")

with checked("B1.13"):
    # the probes on the cached embeddings of the reference backbone (the same as yours, computed once)
    E_train_13 = reference_embeddings(X_TRAIN, SIZE, "train")
    E_test_13 = reference_embeddings(X_TEST, SIZE, "test")
    for name_13, fn_13, kwargs_13, want_13 in (
            ("probe_13", probe_13, {}, reference_probe(E_train_13, Y_TRAIN, E_test_13, Y_TEST)),
            ("probe_13 avec C=0,01", probe_13, {"C": 0.01}, reference_probe(E_train_13, Y_TRAIN, E_test_13, Y_TEST, C=0.01)),
            ("knn_13", knn_13, {}, None), ("knn_13 avec k=1", knn_13, {"k": 1}, None)):
        if want_13 is None:
            from sklearn.neighbors import KNeighborsClassifier
            want_13 = float(KNeighborsClassifier(n_neighbors=kwargs_13.get("k", 10), metric="cosine")
                            .fit(E_train_13, Y_TRAIN).score(E_test_13, Y_TEST))
        got_13 = fn_13(E_train_13.copy(), Y_TRAIN.copy(), E_test_13.copy(), Y_TEST.copy(), **kwargs_13)
        if not returned("B1.13", name_13, got_13):
            continue
        if isinstance(got_13, bool) or not isinstance(got_13, (float, np.floating)) or not 0 <= got_13 <= 1:
            print(f"❌ Ex B1.13 : {name_13} doit rendre une accuracy (un float entre 0 et 1), reçu {got_13!r}.")
            continue
        verdict("B1.13", abs(got_13 - want_13) < 0.003, f"{name_13} : {pct(got_13)} sur les {len(Y_TEST)} images de test.",
                f"{name_13} : {pct(got_13)}, attendu {pct(want_13)} : relis le contrat (standardisation, C, "
                "similarité cosinus, k).")

In [ ]:
# Choose C and k on the VALIDATION images (never on the test), then measure the chosen pair once on the test
with checked("B1.13"):
    E_val_13 = reference_embeddings(X_VAL, SIZE, "val")
    E_train_13 = reference_embeddings(X_TRAIN, SIZE, "train")
    E_test_13 = reference_embeddings(X_TEST, SIZE, "test")
    val_probe_13 = {C: probe_13(E_train_13, Y_TRAIN, E_val_13, Y_VAL, C=C) for C in (0.01, 0.1, 1.0)}
    val_knn_13 = {k: knn_13(E_train_13, Y_TRAIN, E_val_13, Y_VAL, k=k) for k in (1, 5, 10, 25)}
    print("validation, linear probe :", " ; ".join(f"C = {fr(C, 2)} : {pct(a)}" for C, a in val_probe_13.items()))
    print("validation, k-NN cosinus :", " ; ".join(f"k = {k} : {pct(a)}" for k, a in val_knn_13.items()))
    best_C_13 = max(val_probe_13, key=val_probe_13.get)
    best_k_13 = max(val_knn_13, key=val_knn_13.get)
    print(f"test : linear probe (C = {fr(best_C_13, 2)}) {pct(probe_13(E_train_13, Y_TRAIN, E_test_13, Y_TEST, C=best_C_13))} ; "
          f"k-NN (k = {best_k_13}) {pct(knn_13(E_train_13, Y_TRAIN, E_test_13, Y_TEST, k=best_k_13))}")

### Ex B1.14 — Résolution d'entrée : accuracy du linear probe contre temps de calcul (32 à 224 pixels) 🔬 ★★★ ⏱️ 40 min
**Objectif :** mesurer ce que la résolution d'entrée apporte au linear probe et ce qu'elle coûte en temps de calcul, pour en choisir une.  
**Prérequis :** Ex B1.13, B1.12 · fiche §201.5 (la résolution), §201.9 (mesurer une latence) · **Fil rouge :** CIFAR-10 · **Parcours :** C

**Contrat** de `resolution_sweep_14(backbone, sizes, train, test, repeats=3)` : `train` et `test` sont des couples `(images, labels)` ; pour chaque taille de `sizes`, dans l'ordre reçu, un dictionnaire `{"size": taille, "accuracy": ..., "ms_per_image": ...}` :
- `accuracy` : ton `probe_13` sur les embeddings de ton `embed_13` à cette taille ;
- `ms_per_image` : le temps de `embed_13` sur les 64 premières images de test à cette taille, la **plus petite** de `repeats` mesures (`time.perf_counter()`), divisée par 64, en millisecondes.

La cellule de vérification essaie ta fonction sur un petit jeu, avec un backbone dont le premier appel à chaque taille est artificiellement lent (un démarrage à froid). La cellule suivante balaie six tailles, de 32 à 224 pixels, avec `SWEEP_PER_CLASS` images par classe en entraînement et en test (≈ 45 s en mode rapide), puis trace l'accuracy selon la taille et selon le temps par image.

Dans tes notes : décris les deux courbes. Où l'accuracy cesse-t-elle de beaucoup monter ? Le temps suit-il le nombre de pixels ($(224/96)^2 \approx 5{,}4$) ? Quelle résolution choisirais-tu pour un service qui doit répondre en moins de 10 ms par image sur ce CPU, et laquelle pour la meilleure accuracy ? Pourquoi garder la plus petite des mesures, et pas la moyenne ?

In [ ]:
def resolution_sweep_14(backbone, sizes, train, test, repeats=3):
    """For each size of `sizes`, in that order, a dict {"size": size, "accuracy": float, "ms_per_image": float}.

    train and test are pairs (images uint8 (N, 32, 32, 3), labels). accuracy: probe_13 on the embeddings of embed_13
    at that size; ms_per_image: the time of embed_13 on the first 64 test images, the SMALLEST of `repeats`
    measures, divided by 64, in milliseconds."""
    raise NotImplementedError("resolution_sweep_14")

In [ ]:
class SlowFirstCall_14(nn.Module):
    """A probe around a backbone: the first call at each image size waits 0.5 s more (a cold start); it also times
    its other calls, image by image."""

    def __init__(self, inner):
        super().__init__()
        self.inner, self.seen, self.fast_ms = inner, set(), {}

    def forward(self, x):
        if x.shape[-1] not in self.seen:
            self.seen.add(x.shape[-1])
            time.sleep(0.5)
            return self.inner(x)
        start = time.perf_counter()
        out = self.inner(x)
        ms = 1000 * (time.perf_counter() - start) / len(x)
        self.fast_ms[x.shape[-1]] = min(ms, self.fast_ms.get(x.shape[-1], ms))
        return out


with checked("B1.14"):
    small_tr_14 = (X_TRAIN[nested(10)], Y_TRAIN[nested(10)])
    small_te_14 = (X_TEST[nested(10, "test")], Y_TEST[nested(10, "test")])
    probe_net_14 = SlowFirstCall_14(RESNET.model)
    rows_14 = resolution_sweep_14(probe_net_14, [48, 32], small_tr_14, small_te_14, repeats=3)
    if not returned("B1.14", "resolution_sweep_14", rows_14):
        raise StopCheck
    ok_14 = (isinstance(rows_14, list) and len(rows_14) == 2 and all(isinstance(r, dict) for r in rows_14)
             and all({"size", "accuracy", "ms_per_image"} <= set(r) for r in rows_14))
    if not ok_14:
        print('❌ Ex B1.14 : attendu une liste de deux dictionnaires {"size", "accuracy", "ms_per_image"} pour deux '
              f"tailles, reçu {type(rows_14).__name__} {rows_14!r:.200}.")
        raise StopCheck
    verdict("B1.14", [r["size"] for r in rows_14] == [48, 32], "une ligne par taille, dans l'ordre reçu.",
            f"tailles rendues {[r['size'] for r in rows_14]}, attendu [48, 32] (l'ordre reçu).")
    for r_14 in rows_14:
        want_14 = reference_probe(RESNET.embed(small_tr_14[0], r_14["size"]), small_tr_14[1],
                                  RESNET.embed(small_te_14[0], r_14["size"]), small_te_14[1])
        acc_ok_14 = isinstance(r_14["accuracy"], (float, np.floating)) and abs(float(r_14["accuracy"]) - want_14) < 0.011
        verdict("B1.14", acc_ok_14,
                f"{r_14['size']} pixels : accuracy {pct(float(r_14['accuracy']))}, comme le linear probe de référence.",
                f"{r_14['size']} pixels : accuracy {r_14['accuracy']!r:.40}, attendu {pct(want_14)} (probe_13 sur les "
                "embeddings de embed_13 à cette taille).")
    for r_14 in rows_14:
        ms_14, fast_14 = float(r_14["ms_per_image"]), probe_net_14.fast_ms.get(r_14["size"])
        if fast_14 is None:
            print(f"❌ Ex B1.14 : {r_14['size']} pixels : le backbone n'a été appelé qu'une fois à cette taille : "
                  "mesure plusieurs fois.")
            continue
        verdict("B1.14", 0.5 * fast_14 <= ms_14 <= 1.5 * fast_14 + 1.0,
                f"{r_14['size']} pixels : {fr(ms_14, 2)} ms par image ; le démarrage lent du premier appel n'est pas compté.",
                f"{r_14['size']} pixels : {fr(ms_14, 4)} ms par image, alors qu'une passe chaude de ce backbone prend "
                f"{fr(fast_14, 2)} ms par image et que le premier appel à chaque taille dure ici 0,5 s de plus : garde "
                "la plus petite de plusieurs mesures, en millisecondes par image.")

In [ ]:
SWEEP_SIZES = (32, 64, 96, 128, 160, 224)
SWEEP_PER_CLASS = 30 if FAST_MODE else 200     # train and test images per class of the sweep
with checked("B1.14"):
    need("B1.14", "resolution_sweep_14", "make_backbone_13")
    tr_14, te_14 = nested(SWEEP_PER_CLASS), nested(SWEEP_PER_CLASS, "test")
    sweep_14 = resolution_sweep_14(make_backbone_13(), SWEEP_SIZES, (X_TRAIN[tr_14], Y_TRAIN[tr_14]),
                                   (X_TEST[te_14], Y_TEST[te_14]))
    print(f"{'pixels':>6} {'accuracy':>9} {'ms par image':>13}")
    for r_14 in sweep_14:
        print(f"{r_14['size']:>6} {pct(r_14['accuracy']):>9} {fr(r_14['ms_per_image'], 2):>13}")
    acc_14 = np.array([r["accuracy"] for r in sweep_14])
    ms_14 = np.array([r["ms_per_image"] for r in sweep_14])
    fig_14, axes_14 = plt.subplots(1, 2, figsize=(10, 3.6))
    axes_14[0].plot(SWEEP_SIZES, acc_14, "o-")
    axes_14[0].set(xlabel="côté des images (pixels)", ylabel="accuracy du linear probe", title="Accuracy selon la résolution")
    axes_14[1].plot(ms_14, acc_14, "o-")
    for s_14, m_14, a_14 in zip(SWEEP_SIZES, ms_14, acc_14):
        axes_14[1].annotate(str(s_14), (m_14, a_14), textcoords="offset points", xytext=(4, -10), fontsize=8)
    axes_14[1].set(xscale="log", xlabel="temps par image (ms, échelle log)", ylabel="accuracy du linear probe",
                   title="Accuracy contre coût")
    plt.tight_layout()
    plt.show()
    verdict("B1.14", acc_14[-1] - acc_14[0] > 0.1 and ms_14[-1] > ms_14[1],
            "l'accuracy et le temps montent avec la résolution.",
            "ni l'accuracy ni le temps ne montent avec la résolution : relis ton code avant de conclure.")

### Ex B1.15 — Lire des projections PCA/UMAP : pixels bruts, ResNet et DINOv2 📈 ★★ ⏱️ 20 min
**Objectif :** lire des projections en deux dimensions de trois représentations des mêmes images, et dire ce qu'elles montrent et ce qu'elles ne prouvent pas.  
**Prérequis :** Ex B1.13 · ch. 12 (PCA, UMAP) · fiche §201.3 (regarder et sonder), §201.8 (DINOv2) · **Fil rouge :** CIFAR-10 · **Parcours :** C

La cellule suivante projette en deux dimensions les mêmes 600 images de test (60 par classe), sous trois représentations : (A) les **pixels bruts** (3 072 valeurs), (B) les embeddings de **ResNet-18** (512), (C) ceux de **DINOv2-small** (384, images à `DINO_SIZE` pixels), chacune standardisée, puis réduite par une **PCA** (en haut) et par **UMAP** (en bas, `n_neighbors=15`, graine fixée). Une couleur par classe (≈ 1 min 30 s la première fois : les embeddings DINOv2 de tout `X_TEST` sont calculés, puis mis en cache pour la suite du notebook ; ≈ 25 s ensuite).

Lis la figure et écris :
a) la lettre de la représentation dont l'UMAP montre les groupes **les plus nets** : `"A"`, `"B"` ou `"C"` ;
b) `True` ou `False` : les groupes que l'on voit dans l'UMAP des pixels bruts (A) sont des classes ;
c) `True` ou `False` : dans l'UMAP de (B), les quatre véhicules (avion, automobile, bateau, camion) occupent une région à part des six animaux ;
d) `True` ou `False` : la PCA de (B) mêle les classes, donc un classifieur linéaire sur ces embeddings ne peut pas faire beaucoup mieux que le hasard.

Une fois tes quatre réponses écrites, la cellule de vérification affiche des mesures faites sur les projections : l'accuracy des 10 plus proches voisins **dans le plan** de chaque projection, et la part des images classées dans la bonne famille (véhicules ou animaux).

Dans tes notes : pourquoi la PCA des pixels ne sépare-t-elle presque rien (que varie-t-il le plus d'une photo à l'autre) ? Pourquoi UMAP sépare-t-il mieux que la PCA, et pourquoi ne faut-il pas lire les **distances entre** les groupes d'un UMAP (ch. 12) ?

In [ ]:
# Six projections of the same 600 test images (60 per class): PCA and UMAP of (A) the raw pixels, (B) the embeddings
# of ResNet-18 (SIZE pixels) and (C) those of DINOv2-small (DINO_SIZE pixels); one colour per class
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

wb.ensure("umap-learn")
with warnings.catch_warnings():
    warnings.simplefilter("ignore")        # umap imports tqdm, which warns about Jupyter widgets
    import umap

DINO = transfer.try_backbone("dinov2_small", "B1.15")
idx_15 = nested(60, "test")
y_15 = Y_TEST[idx_15]
REPS_15 = {"(A) pixels bruts": X_TEST[idx_15].reshape(len(idx_15), -1) / 255.0,
           "(B) ResNet-18": reference_embeddings(X_TEST, SIZE, "test")[idx_15]}
if DINO is not None:   # the embeddings of the whole X_TEST, computed once (≈ 1 min) and reused by parts C and D
    DINO_TEST = transfer.cached_embeddings(DINO, X_TEST, size=DINO_SIZE, subset="test", verbose=False)
    REPS_15["(C) DINOv2-small"] = DINO_TEST[idx_15]
PROJ_15 = {}
for name_15, E_15 in REPS_15.items():
    Z_15 = StandardScaler().fit_transform(E_15)
    PROJ_15[name_15] = {"PCA": PCA(2, svd_solver="full").fit_transform(Z_15),
                        "UMAP": umap.UMAP(n_neighbors=15, min_dist=0.1, random_state=0, n_jobs=1).fit_transform(Z_15)}
fig_15, axes_15 = plt.subplots(2, len(PROJ_15), figsize=(4.2 * len(PROJ_15), 8.2), squeeze=False)
colors_15 = plt.cm.tab10(np.arange(10))
for j_15, (name_15, proj_15) in enumerate(PROJ_15.items()):
    for i_15, method_15 in enumerate(("PCA", "UMAP")):
        ax_15 = axes_15[i_15, j_15]
        for c_15 in range(10):
            pts_15 = proj_15[method_15][y_15 == c_15]
            ax_15.scatter(pts_15[:, 0], pts_15[:, 1], s=7, color=colors_15[c_15], label=CLASSES[c_15])
        ax_15.set_title(f"{name_15} — {method_15}", fontsize=10)
        ax_15.set_xticks([])
        ax_15.set_yticks([])
axes_15[0, -1].legend(fontsize=7, markerscale=2, loc="upper left", bbox_to_anchor=(1.0, 1.0))
plt.tight_layout()
plt.show()

In [ ]:
separated_15 = ...       # a) the letter of the representation whose UMAP shows the best-separated groups: "A", "B" or "C"
pixel_classes_15 = ...   # b) True or False: the groups seen in the UMAP of the raw pixels (A) are classes
vehicles_15 = ...        # c) True or False: in the UMAP of (B), the four vehicles (airplane, automobile, ship, truck)
                         #    form a region apart from the six animals
pca_says_15 = ...        # d) True or False: the PCA of (B) mixes the classes, so a linear classifier on these
                         #    embeddings cannot do much better than chance

In [ ]:
for id_15, value_15 in (("B1.15a", separated_15), ("B1.15b", pixel_classes_15), ("B1.15c", vehicles_15), ("B1.15d", pca_says_15)):
    with wb.attempt(id_15):
        wb.check(id_15, value_15)

In [ ]:
if not filled(separated_15, pixel_classes_15, vehicles_15, pca_says_15):
    print("⏳ Ex B1.15 : les mesures s'affichent une fois tes quatre réponses écrites.")
else:
    # what the figure was showing, measured: the accuracy of the 10 nearest neighbours IN the plane of each projection
    # (5-fold cross-validation), and the share of images put in the right family (vehicles or animals)
    from sklearn.model_selection import cross_val_predict
    from sklearn.neighbors import KNeighborsClassifier
    vehicle_15 = np.isin(y_15, [0, 1, 8, 9])
    for name_15, proj_15 in PROJ_15.items():
        line_15 = []
        for method_15 in ("PCA", "UMAP"):
            pred_15 = cross_val_predict(KNeighborsClassifier(10), proj_15[method_15], y_15, cv=5)
            family_15 = np.mean(np.isin(pred_15, [0, 1, 8, 9]) == vehicle_15)
            line_15.append(f"{method_15} : 10-NN dans le plan {pct(np.mean(pred_15 == y_15), 0)}, bonne famille {pct(family_15, 0)}")
        print(f"{name_15} — " + " ; ".join(line_15))
    print("(le hasard : 10 % pour la classe ; le linear probe de B1.13, sur les 512 dimensions de (B) : plus de 70 %)")

### Ex B1.16 — Sonder chaque bloc : linear probe couche par couche, CIFAR-10 contre Fashion-MNIST 🔬 ★★★ ⏱️ 45 min
**Objectif :** mesurer, bloc par bloc, ce qu'un backbone ImageNet garde d'utile pour un domaine proche et pour un domaine lointain.  
**Prérequis :** Ex B1.13, 📄 B1.10 · fiche §201.1 (hiérarchie des features), §201.3 (regarder et sonder), §201.5 (les canaux), §201.7 (distance de domaine) · **Fil rouge :** CIFAR-10 · **Parcours :** M, C

Le 📄 B1.10 a montré, sur ImageNet coupé en deux, jusqu'où les features se transfèrent. Ici, deux tâches : CIFAR-10 (des photos, proches d'ImageNet) et Fashion-MNIST (des vêtements en niveaux de gris sur fond noir, loin d'ImageNet). La cellule suivante charge les deux et donne `to_input(images, size)` (de `wb.transfer`) : la préparation de ton `embed_13`, qui accepte aussi des images en niveaux de gris `(N, H, W)`, dont le canal est répété trois fois après l'agrandissement (fiche §201.5).

**Contrat**
- `block_embeddings_16(model, images, size, blocks=RESNET_BLOCKS, batch_size=64)` : pour un ResNet-18 de torchvision en mode `eval()` et des images `uint8` (en couleur ou en niveaux de gris), un dictionnaire `{bloc: tableau float32 (N, C)}` : le **pooling moyen global** de la sortie de chaque bloc demandé, parmi `"stem"` (la sortie de `model.maxpool`), `"layer1"` … `"layer4"`. Chaque mini-batch est préparé par `to_input(batch, size)` et ne passe qu'**une fois** dans le réseau pour tous les blocs, sans gradient ;
- `probe_by_block_16(model, train, test, size, blocks=RESNET_BLOCKS)` : `{bloc: accuracy de test de ton probe_13 sur les embeddings de ce bloc}`.

La cellule de vérification compare tes embeddings à ceux de `RESNET` sur des images des deux datasets et compte les passes dans le réseau. La suivante sonde les cinq blocs sur les deux datasets, avec `BLOCK_PER_CLASS` images par classe (≈ 15 s en mode rapide), et trace les deux courbes.

Dans tes notes : sur CIFAR-10, quel bloc donne le meilleur probe ? Et sur Fashion-MNIST ? Compare l'écart entre le stem et le meilleur bloc sur chaque dataset : que dit-il de la distance de domaine ? Relie tes courbes au « général → spécifique » de Yosinski et coll. (📄 B1.10). Quelle couche choisirais-tu comme embedding pour une tâche loin d'ImageNet, avant tout fine-tuning ?

In [ ]:
# Two datasets for the probes, block by block: CIFAR-10 (close to ImageNet) and Fashion-MNIST (far: grayscale clothes)
from wb.transfer import RESNET_BLOCKS, to_input   # to_input(images, size): what embed_13 does, grayscale accepted

BLOCK_PER_CLASS = 50 if FAST_MODE else 300          # train and test images per class
XC_TR_16, YC_TR_16 = X_TRAIN[nested(BLOCK_PER_CLASS)], Y_TRAIN[nested(BLOCK_PER_CLASS)]
XC_TE_16, YC_TE_16 = X_TEST[nested(BLOCK_PER_CLASS, "test")], Y_TEST[nested(BLOCK_PER_CLASS, "test")]
XF_TR_16, YF_TR_16 = wb.datasets.load_fashion_mnist_subset(BLOCK_PER_CLASS, "train")   # uint8 (N, 28, 28)
XF_TE_16, YF_TE_16 = wb.datasets.load_fashion_mnist_subset(BLOCK_PER_CLASS, "test")
print("blocs :", RESNET_BLOCKS, "; to_input(XF_TR_16[:2], SIZE).shape =", tuple(to_input(XF_TR_16[:2], SIZE).shape))

In [ ]:
def block_embeddings_16(model, images, size, blocks=RESNET_BLOCKS, batch_size=64):
    """{block: float32 NumPy array (N, C)}: the global average pooling of the output of each block of `blocks`.

    model: a torchvision ResNet-18 in eval() mode (its head is not used); images: uint8, in colour (N, H, W, 3) or
    grayscale (N, H, W), each mini-batch prepared by to_input(batch, size); "stem" is the output of model.maxpool.
    ONE forward pass per mini-batch for all the blocks, without gradient."""
    raise NotImplementedError("block_embeddings_16")


def probe_by_block_16(model, train, test, size, blocks=RESNET_BLOCKS):
    """{block: test accuracy of probe_13 on the embeddings of that block}; train and test are pairs (images, labels)."""
    raise NotImplementedError("probe_by_block_16")

In [ ]:
with checked("B1.16"):
    calls_16 = []
    hook_16 = RESNET.full.conv1.register_forward_hook(lambda m, i, o: calls_16.append(o.shape[0]))
    try:
        for name_16, sample_16 in (("CIFAR-10", X_TEST[:6].copy()), ("Fashion-MNIST", XF_TE_16[:6].copy())):
            want_16 = RESNET.embed_blocks(sample_16, 64, batch_size=4)
            calls_16.clear()                             # count the passes of YOUR function only
            got_16 = block_embeddings_16(RESNET.full, sample_16, 64, batch_size=4)
            passes_16 = list(calls_16)
            if not returned("B1.16", "block_embeddings_16", got_16):
                raise StopCheck
            if not isinstance(got_16, dict) or set(got_16) != set(RESNET_BLOCKS):
                print(f"❌ Ex B1.16 : attendu un dictionnaire aux clés {list(RESNET_BLOCKS)}, reçu "
                      f"{sorted(got_16) if isinstance(got_16, dict) else type(got_16).__name__}.")
                raise StopCheck
            ok_16 = all(same("B1.16", f"{name_16}, bloc {b}", got_16[b], want_16[b], rtol=1e-4, atol=1e-4)
                        for b in RESNET_BLOCKS)
            verdict("B1.16", passes_16 == [4, 2], f"{name_16} : une seule passe avant par mini-batch pour les cinq blocs.",
                    f"{name_16} : {len(passes_16)} passes dans conv1 pour 6 images en mini-batches de 4 ({passes_16}) : "
                    "attendu une passe par mini-batch, tous les blocs à la fois.")
    finally:
        hook_16.remove()
    tr_16, te_16 = nested(10), nested(10, "test")
    probes_16 = probe_by_block_16(RESNET.full, (X_TRAIN[tr_16], Y_TRAIN[tr_16]), (X_TEST[te_16], Y_TEST[te_16]), 64,
                                  blocks=("stem", "layer4"))
    if not returned("B1.16", "probe_by_block_16", probes_16):
        raise StopCheck
    if not isinstance(probes_16, dict) or set(probes_16) != {"stem", "layer4"}:
        print(f"❌ Ex B1.16 : probe_by_block_16(..., blocks=('stem', 'layer4')) doit rendre un dictionnaire à ces deux "
              f"clés, reçu {probes_16!r:.120}.")
        raise StopCheck
    for b_16 in ("stem", "layer4"):
        want_16 = reference_probe(RESNET.embed(X_TRAIN[tr_16], 64, b_16), Y_TRAIN[tr_16],
                                  RESNET.embed(X_TEST[te_16], 64, b_16), Y_TEST[te_16])
        verdict("B1.16", abs(float(probes_16[b_16]) - want_16) < 0.011,
                f"probe_by_block_16, bloc {b_16} : {pct(float(probes_16[b_16]))}, comme la référence.",
                f"probe_by_block_16, bloc {b_16} : "
                + (pct(float(probes_16[b_16])) if isinstance(probes_16[b_16], (float, np.floating)) else repr(probes_16[b_16]))
                + f", attendu {pct(want_16)}.")

In [ ]:
with checked("B1.16"):
    need("B1.16", "probe_by_block_16")
    model_16 = make_backbone_13()
    results_16 = {"CIFAR-10": probe_by_block_16(model_16, (XC_TR_16, YC_TR_16), (XC_TE_16, YC_TE_16), SIZE),
                  "Fashion-MNIST": probe_by_block_16(model_16, (XF_TR_16, YF_TR_16), (XF_TE_16, YF_TE_16), SIZE)}
    print(f"{'bloc':>8} " + " ".join(f"{name:>14}" for name in results_16))
    for b_16 in RESNET_BLOCKS:
        print(f"{b_16:>8} " + " ".join(f"{pct(results_16[name][b_16]):>14}" for name in results_16))
    plt.figure(figsize=(7, 3.8))
    for name_16, accs_16 in results_16.items():
        plt.plot(list(RESNET_BLOCKS), [accs_16[b] for b in RESNET_BLOCKS], "o-", label=name_16)
    plt.ylabel("accuracy du linear probe (test)")
    plt.xlabel("bloc sondé (sortie, pooling moyen global)")
    plt.title(f"Linear probe bloc par bloc (ResNet-18 gelé, {BLOCK_PER_CLASS} images par classe)", fontsize=10)
    plt.legend()
    plt.tight_layout()
    plt.show()

## Partie B · Fine-tuner : le dernier bloc, puis tout le réseau

On ne gèle plus tout le backbone : on dégèle le dernier bloc, puis tout le réseau, avec des learning rates par groupe, un planning et des batchnorms maîtrisées. En mode rapide : `FT_PER_CLASS` = 50 images par classe (500 images), 96 pixels, mini-batches de 32, `FT_EPOCHS` = 3 epochs ; un pas de fine-tuning complet de ResNet-18 coûte alors ≈ 0,6 s sur deux cœurs. 🚀 En mode complet (`FAST_MODE = False`) : 1 000 images par classe à 224 pixels et 10 epochs, à lancer sur un GPU de Colab. La cellule suivante donne `batches(X, y, batch_size, seed)` (des mini-batches mélangés, préparés et normalisés), `evaluate(model, X, y)` (l'accuracy, en mode `eval()`) et `IMAGENET_RESNET18` (les poids pré-entraînés, pour les vérifications).

In [ ]:
# The data and the helpers of part B: fine-tuning ResNet-18 on CIFAR-10
from wb.transfer import to_input

FT_PER_CLASS = 50 if FAST_MODE else 1000     # training images per class of the fine-tunings (nested in X_TRAIN)
FT_EPOCHS = 3 if FAST_MODE else 10
XT_B, YT_B = X_TRAIN[nested(FT_PER_CLASS)], Y_TRAIN[nested(FT_PER_CLASS)]
IMAGENET_RESNET18 = RESNET.full.state_dict()       # the pre-trained weights, ImageNet head included (for the checks)


def batches(X, y, batch_size, seed):
    """Shuffled mini-batches (x, y) of uint8 images X: x prepared by to_input(..., SIZE) (normalised), both on DEVICE.
    The order depends on `seed` only."""
    order = torch.randperm(len(X), generator=torch.Generator().manual_seed(seed)).numpy()
    for i in range(0, len(X), batch_size):
        idx = order[i:i + batch_size]
        yield to_input(X[idx], SIZE).to(DEVICE), torch.from_numpy(y[idx]).to(DEVICE)


def evaluate(model, X, y, batch_size=128):
    """Accuracy of `model` on uint8 images X at SIZE pixels, in eval() mode, without gradient (the model stays in
    eval() mode afterwards)."""
    model.eval()
    correct = 0
    with torch.inference_mode():
        for i in range(0, len(X), batch_size):
            pred = model(to_input(X[i:i + batch_size], SIZE).to(DEVICE)).argmax(1).cpu().numpy()
            correct += int((pred == y[i:i + batch_size]).sum())
    return correct / len(X)


def changed(a, b):
    """True when two tensors differ."""
    return not torch.equal(a.detach().cpu(), b.detach().cpu())


print(f"fine-tuning : {len(XT_B)} images d'entraînement ({FT_PER_CLASS} par classe), {len(X_VAL)} de validation, "
      f"{FT_EPOCHS} epochs au plus, {SIZE} pixels")

### Ex B1.17 — Fine-tuning partiel : dégeler le dernier bloc avec des learning rates par groupe 📦 ★★★ ⏱️ 45 min
**Objectif :** fine-tuner le dernier bloc d'un ResNet pré-entraîné avec des learning rates par groupe, en gardant les batchnorms sur leurs statistiques d'ImageNet.  
**Prérequis :** Ex B1.13 · ✏️ B1.3, B1.4 · ch. 23 (gel, nouvelle tête) · fiche §201.4 · **Fil rouge :** CIFAR-10 · **Parcours :** R, C

**Contrat**
- `param_groups_17(modules, head_lr, gamma, weight_decay=0.01)` : les groupes de paramètres d'AdamW. `modules` est une liste de modules **de la tête vers l'entrée** (par exemple `[model.fc, model.layer4]`) ; le $k$-ième (à partir de 0) reçoit le learning rate `head_lr * gamma**k`. Chaque module donne au plus deux dictionnaires `{"params", "lr", "weight_decay"}` : ses paramètres de dimension 2 ou plus avec `weight_decay`, les autres (biais, paramètres des batchnorms) avec 0. Seulement les paramètres qui demandent un gradient, et aucun groupe vide ;
- `make_partial_17(num_classes=10)` : ResNet-18 `IMAGENET1K_V1`, une nouvelle tête `nn.Linear(512, num_classes)`, tous les paramètres gelés sauf ceux de `layer4` et de la tête ; sur `DEVICE` ;
- `freeze_batchnorm_17(model)` : met chaque batchnorm de `model` en mode `eval()`, sans changer le mode des autres modules ;
- `train_partial_17(model, X, y, epochs, head_lr, gamma, batch_size=32, seed=0)` : crée **un** `torch.optim.AdamW` sur `param_groups_17([model.fc, model.layer4], head_lr, gamma)`, puis, pour chaque epoch : `model.train()`, puis `freeze_batchnorm_17(model)`, puis les mini-batches de `batches(X, y, batch_size, seed + epoch)` avec la cross-entropy. Rend la liste des losses d'entraînement moyennes de chaque epoch (des `float`).

La cellule de vérification essaie tes groupes sur un petit réseau, contrôle ton modèle (ce qui s'entraîne, les poids de départ, les modes), puis fait deux pas d'entraînement : elle observe les learning rates que reçoit AdamW, les blocs gelés, les statistiques glissantes des batchnorms, la tête et `layer4`. La cellule suivante fine-tune `FT_EPOCHS` epochs sur `XT_B` (≈ 30 s en mode rapide) avec `head_lr = 1e-3` et `gamma = 0,1`, et compare au linear probe sur les **mêmes** images.

Dans tes notes : combien de paramètres entraînes-tu (✏️ B1.4) ? Comment évoluent la loss d'entraînement et l'accuracy de test : le dégel de `layer4` rapporte-t-il beaucoup par rapport à la tête seule, avec 50 images par classe ? Pourquoi aucun weight decay sur les biais et les paramètres des batchnorms ?

In [ ]:
def param_groups_17(modules, head_lr, gamma, weight_decay=0.01):
    """Parameter groups for AdamW. modules: a list of modules from the head DOWN (e.g. [model.fc, model.layer4]); the
    k-th one gets the learning rate head_lr * gamma**k. Each module gives (at most) two dicts {"params", "lr",
    "weight_decay"}: its parameters of dimension 2 or more with weight_decay, the others (biases, batchnorm parameters)
    with 0. Only the parameters that require a gradient; no empty group."""
    raise NotImplementedError("param_groups_17")


def make_partial_17(num_classes=10):
    """ResNet-18 IMAGENET1K_V1 with a new head nn.Linear(512, num_classes); every parameter frozen except those of
    layer4 and of the head. On DEVICE."""
    raise NotImplementedError("make_partial_17")


def freeze_batchnorm_17(model):
    """Put every batchnorm of `model` in eval() mode (the other modules keep their mode)."""
    raise NotImplementedError("freeze_batchnorm_17")


def train_partial_17(model, X, y, epochs, head_lr, gamma, batch_size=32, seed=0):
    """Fine-tunes the model of make_partial_17 on (X, y) and returns the mean training loss of each epoch (floats)."""
    raise NotImplementedError("train_partial_17")

In [ ]:
class SpyAdamW_17(torch.optim.AdamW):
    """torch.optim.AdamW, which also remembers the learning rate of each of its groups (for the check)."""
    seen = []

    def __init__(self, params, *args, **kwargs):
        super().__init__(params, *args, **kwargs)
        SpyAdamW_17.seen.append([(g["lr"], g["weight_decay"], [id(p) for p in g["params"]]) for g in self.param_groups])


def group_report_17(groups, modules, head_lr, gamma, weight_decay):
    """What is wrong with a list of parameter groups ("" when it follows the contract)."""
    if not isinstance(groups, list) or not all(isinstance(g, dict) and {"params", "lr", "weight_decay"} <= set(g)
                                               for g in groups):
        return 'attendu une liste de dictionnaires {"params", "lr", "weight_decay"}'
    owner = {}
    for k, module in enumerate(modules):
        for p in module.parameters():
            if p.requires_grad:
                owner[id(p)] = (k, p.ndim)
    seen = []
    for g in groups:
        params = list(g["params"])
        if not params:
            return "un groupe est vide"
        for p in params:
            if id(p) not in owner:
                return "un groupe contient un paramètre gelé ou d'un autre module"
            k, ndim = owner[id(p)]
            if not math.isclose(g["lr"], head_lr * gamma ** k, rel_tol=1e-9):
                return f"le module n° {k} (en partant de la tête) a le learning rate {g['lr']:.3g}, attendu head_lr × gamma^{k}"
            want = weight_decay if ndim >= 2 else 0.0
            if not math.isclose(g["weight_decay"], want, abs_tol=1e-12):
                return (f"un paramètre de dimension {ndim} a weight_decay={g['weight_decay']}, attendu {want} "
                        "(les biais et les paramètres des batchnorms n'en ont pas)")
            seen.append(id(p))
    if len(seen) != len(set(seen)):
        return "un paramètre est dans deux groupes"
    if set(seen) != set(owner):
        return f"{len(set(owner) - set(seen))} paramètre(s) entraînable(s) dans aucun groupe"
    return ""


with checked("B1.17"):
    toy_17 = nn.Sequential(nn.Linear(4, 3), nn.BatchNorm1d(3), nn.Linear(3, 2))
    toy_17[0].bias.requires_grad_(False)
    modules_17 = [toy_17[2], nn.ModuleList([toy_17[0], toy_17[1]])]
    groups_17 = param_groups_17(modules_17, 1e-3, 0.5, weight_decay=0.05)
    if not returned("B1.17", "param_groups_17", groups_17):
        raise StopCheck
    problem_17 = group_report_17(groups_17, modules_17, 1e-3, 0.5, 0.05)
    verdict("B1.17", not problem_17, "param_groups_17 : learning rates par module, weight decay sur les poids seulement, "
            "paramètres gelés exclus.", f"param_groups_17 sur un petit réseau : {problem_17}.")
    if problem_17:
        raise StopCheck

with checked("B1.17"):
    model_17 = make_partial_17()
    if not returned("B1.17", "make_partial_17", model_17):
        raise StopCheck
    head_ok_17 = isinstance(getattr(model_17, "fc", None), nn.Linear) and model_17.fc.out_features == 10
    verdict("B1.17", head_ok_17, "une nouvelle tête à 10 sorties.", "attendu model.fc = nn.Linear(512, 10).")
    trainable_17 = {n for n, p in model_17.named_parameters() if p.requires_grad}
    want_17 = {n for n, _ in model_17.named_parameters() if n.startswith(("layer4.", "fc."))}
    verdict("B1.17", trainable_17 == want_17, "seuls layer4 et la tête sont entraînables.",
            f"{len(trainable_17 - want_17)} paramètre(s) entraînable(s) en trop, {len(want_17 - trainable_17)} gelé(s) "
            "à tort : seuls layer4 et la tête s'entraînent.")
    sd_17 = model_17.state_dict()
    verdict("B1.17", all(torch.equal(sd_17[k].cpu(), v) for k, v in IMAGENET_RESNET18.items() if not k.startswith("fc.")),
            "le backbone a les poids IMAGENET1K_V1.", "le backbone n'a pas les poids IMAGENET1K_V1.")
    model_17.train()
    freeze_batchnorm_17(model_17)
    bns_17 = [m for m in model_17.modules() if isinstance(m, nn.BatchNorm2d)]
    verdict("B1.17", bns_17 and not any(m.training for m in bns_17) and model_17.training and model_17.layer1.training,
            "freeze_batchnorm_17 : les 20 batchnorms en eval(), le reste en train().",
            "après model.train() puis freeze_batchnorm_17(model), attendu les batchnorms en eval() et le reste en train().")

with checked("B1.17"), torch.random.fork_rng():
    model_17 = make_partial_17()                       # a fresh model for two training steps
    if not returned("B1.17", "make_partial_17", model_17):
        raise StopCheck
    head_before_17 = model_17.fc.weight.detach().clone()
    SpyAdamW_17.seen.clear()
    adamw_17, torch.optim.AdamW = torch.optim.AdamW, SpyAdamW_17
    try:
        losses_17 = train_partial_17(model_17, X_TRAIN[nested(2)], Y_TRAIN[nested(2)], 1, 1e-3, 0.1, batch_size=10)
    finally:
        torch.optim.AdamW = adamw_17
    if not returned("B1.17", "train_partial_17", losses_17):
        raise StopCheck
    verdict("B1.17", isinstance(losses_17, list) and len(losses_17) == 1 and all(isinstance(v, float) and math.isfinite(v) for v in losses_17),
            f"train_partial_17 rend une loss par epoch ({fr(losses_17[0], 3) if isinstance(losses_17, list) and losses_17 else '?'}).",
            f"attendu une liste d'un float par epoch, reçu {losses_17!r:.100}.")
    if SpyAdamW_17.seen:
        lrs_17 = sorted({round(lr, 12) for lr, _, _ in SpyAdamW_17.seen[-1]})
        verdict("B1.17", lrs_17 == [1e-4, 1e-3], "AdamW reçoit 1e-3 pour la tête et 1e-4 pour layer4 (gamma = 0,1).",
                f"learning rates reçus par AdamW : {lrs_17}, attendu [1e-4, 1e-3] (param_groups_17 avec [model.fc, model.layer4]).")
    else:
        print("⚠️ Ex B1.17 : l'optimiseur n'a pas pu être observé (crée-le par torch.optim.AdamW dans train_partial_17).")
    sd_17 = model_17.state_dict()
    frozen_17 = [k for k, v in IMAGENET_RESNET18.items() if not k.startswith(("layer4.", "fc.")) and "running" not in k
                 and "num_batches" not in k and changed(sd_17[k], v)]
    stats_17 = [k for k, v in IMAGENET_RESNET18.items() if ("running" in k or "num_batches" in k) and changed(sd_17[k], v)]
    verdict("B1.17", not frozen_17, "les blocs gelés n'ont pas bougé.", f"{len(frozen_17)} tenseurs gelés ont changé (par ex. {frozen_17[:1]}).")
    verdict("B1.17", not stats_17, "les statistiques glissantes des batchnorms sont restées celles d'ImageNet.",
            f"{len(stats_17)} statistiques glissantes ont changé (par ex. {stats_17[:1]}) : dans quel mode étaient les "
            "batchnorms pendant l'entraînement ?")
    verdict("B1.17", changed(model_17.fc.weight, head_before_17) and any(changed(sd_17[k], IMAGENET_RESNET18[k]) for k in sd_17 if k.startswith("layer4.") and "running" not in k),
            "la tête et layer4 ont appris.", "la tête ou layer4 n'ont pas bougé : l'optimiseur les reçoit-il ?")

In [ ]:
with checked("B1.17"), torch.random.fork_rng():
    torch.manual_seed(17)
    MODEL_17 = make_partial_17()
    start_17 = time.perf_counter()
    losses_17 = train_partial_17(MODEL_17, XT_B, YT_B, FT_EPOCHS, head_lr=1e-3, gamma=0.1)
    seconds_17 = time.perf_counter() - start_17
    val_17, test_17 = evaluate(MODEL_17, X_VAL, Y_VAL), evaluate(MODEL_17, X_TEST, Y_TEST)
    probe_17 = reference_probe(reference_embeddings(X_TRAIN, SIZE, "train")[nested(FT_PER_CLASS)], YT_B,
                               reference_embeddings(X_TEST, SIZE, "test"), Y_TEST)
    print("loss d'entraînement par epoch :", ", ".join(fr(v, 3) for v in losses_17), f"({seconds_17:.0f} s)")
    print(f"layer4 + tête : validation {pct(val_17)}, test {pct(test_17)} ; tête seule (linear probe sur les mêmes "
          f"{len(XT_B)} images) : test {pct(probe_17)}")

### Ex B1.18 — Le fine-tuning qui détruit le modèle : quatre erreurs à trouver 🐛 ★★ ⏱️ 25 min
**Objectif :** trouver et corriger les quatre erreurs d'un fine-tuning, guidé par des diagnostics enchaînés.  
**Prérequis :** Ex B1.17 · fiche §201.4, §201.5 · tableau des pièges ⚠️ · **Fil rouge :** CIFAR-10 · **Parcours :** R, C

Un collègue a écrit `finetune_18`, un fine-tuning court de **tout** ResNet-18 par mini-batches de 4 images. Le code tourne sans erreur, mais le modèle n'apprend pas ce qu'il devrait, et son réseau pré-entraîné en sort abîmé. La fonction contient **quatre** erreurs. Corrige-la **sur place**, dans sa cellule, sans changer sa signature ni ce qu'elle rend.

La cellule de vérification lance `diagnose_18` : un fine-tuning de 12 pas sur 50 images, observé de l'intérieur, et quatre diagnostics enchaînés, chacun ne s'exécutant que si le précédent passe (une erreur peut en cacher une autre) : la tête apprend-elle ? que reçoit la première convolution ? que deviennent les statistiques glissantes des batchnorms ? de combien bougent les poids de la première couche ? Elle tourne dès le premier « Tout exécuter » : ses messages sont ton point de départ. Une fois les quatre points passés, la cellule suivante entraîne ta version `STEPS_18` pas et mesure son accuracy de test.

Dans tes notes : les quatre erreurs, et pour chacune le symptôme qu'elle produit seule. Lesquelles auraient laissé une loss d'entraînement qui baisse quand même ? Pourquoi l'erreur de la batchnorm ne se voit-elle qu'à l'évaluation ?

In [ ]:
def diagnose_18(fn, steps=12):
    """Four diagnoses of a short fine-tuning, in a row: the first one that fails stops the chain (one error can hide
    another). Returns True when the four pass."""
    images, labels = X_TRAIN[nested(5)], Y_TRAIN[nested(5)]
    record = {"head": None, "inputs": []}

    def spy(module, inputs):                    # sees every module call: the first 3-channel convolution, the new head
        if isinstance(module, nn.Conv2d) and module.in_channels == 3 and module.out_channels == 64:
            record["inputs"].append((float(inputs[0].mean()), float(inputs[0].std())))
        if isinstance(module, nn.Linear) and module.out_features == 10 and record["head"] is None:
            record["head"] = module.weight.detach().clone()

    handle = torch.nn.modules.module.register_module_forward_pre_hook(spy)
    try:
        with torch.random.fork_rng():
            result = fn(images, labels, steps=steps)
    finally:
        handle.remove()
    if not (isinstance(result, tuple) and len(result) == 2):
        print(f"❌ Ex B1.18 : finetune_18 doit rendre le couple (model, losses), reçu {type(result).__name__}.")
        return False
    model, losses = result
    ref = IMAGENET_RESNET18
    head = getattr(model, "fc", None)
    moved_head = (float(np.nan_to_num((head.weight.detach() - record["head"]).abs().max().item(), nan=np.inf))
                  if record["head"] is not None and isinstance(head, nn.Linear)
                  and head.weight.shape == record["head"].shape else 0.0)                 # NaN weights have moved
    mean_in = float(np.mean([m for m, _ in record["inputs"]])) if record["inputs"] else float("nan")
    std_in = float(np.mean([s for _, s in record["inputs"]])) if record["inputs"] else float("nan")
    drift = float((model.bn1.running_mean.detach() - ref["bn1.running_mean"]).abs().max())
    w0 = ref["conv1.weight"]
    rel = float((model.conv1.weight.detach() - w0).norm() / w0.norm())
    last = float(np.mean(losses[-4:])) if len(losses) else float("nan")
    points = [  # (what passes, test, what was measured, a question that points where to look)
        ("la nouvelle tête apprend", moved_head > 1e-6,
         f"plus grand changement d'un poids de la tête à 10 classes en {steps} pas : {fr(moved_head, 6)}",
         "les poids de la tête n'ont pas bougé : quels paramètres l'optimiseur met-il à jour, et depuis quand existe la tête ?"),
        ("la première convolution reçoit des entrées à l'échelle de son pré-entraînement", std_in > 0.6,
         f"entrées de conv1 : moyenne {fr(mean_in, 2)}, écart-type {fr(std_in, 2)}",
         "ce sont des pixels entre 0 et 1 : est-ce ainsi qu'étaient préparées les images sur lesquelles ResNet-18 a appris ?"),
        ("les statistiques glissantes des batchnorms restent celles d'ImageNet", drift < 1e-6,
         f"plus grand écart de bn1.running_mean à sa valeur ImageNet : {fr(drift, 4)}",
         f"bn1.running_mean a dérivé en {steps} pas de 4 images : avec quelles statistiques les batchnorms normalisent-elles "
         "pendant l'entraînement, et que mettent-elles à jour ?"),
        ("les couches pré-entraînées bougent peu et la loss reste finie", rel < 0.02 and math.isfinite(last),
         f"changement relatif des poids de conv1 : {pct(rel, 2)} ; loss moyenne des 4 derniers pas : "
         + (fr(last, 3) if abs(last) < 1e4 else f"{last:.2e}".replace(".", ",")),
         "les premières couches changent beaucoup en quelques pas : quelle taille de pas reçoivent-elles ?"),
    ]
    for k, (name, ok, measured, why) in enumerate(points, start=1):
        if ok:
            print(f"✅ Ex B1.18 : point {k}, {name} ({measured}).")
        else:
            print(f"❌ Ex B1.18 : point {k} : {measured} ; {why}")
            for later in range(k + 1, len(points) + 1):
                print(f"⏳ Ex B1.18 : point {later} : corrige d'abord le point {k}.")
            return False
    print("✅ Ex B1.18 : les quatre diagnostics passent.")
    return True

In [ ]:
# The fine-tuning to repair IN PLACE (same signature, same returns): the diagnostics of the next cell start from it
from torchvision.models import ResNet18_Weights, resnet18


def finetune_18(images, labels, steps=12, batch_size=4, seed=0):
    """A short fine-tuning of the whole ResNet-18 on CIFAR-10 images (uint8 (N, 32, 32, 3)), by mini-batches of
    batch_size images at SIZE pixels. Returns (model, losses): the network in eval() mode, the loss of each step."""
    torch.manual_seed(seed)
    model = resnet18(weights=ResNet18_Weights.IMAGENET1K_V1)
    optimizer = torch.optim.AdamW(model.parameters(), lr=1e-1)
    model.fc = nn.Linear(model.fc.in_features, 10)
    model.train()
    order = torch.randperm(len(images), generator=torch.Generator().manual_seed(seed)).numpy()
    losses = []
    for step in range(steps):
        idx = order[(np.arange(batch_size) + step * batch_size) % len(images)]
        x = torch.from_numpy(images[idx]).permute(0, 3, 1, 2).float() / 255
        x = F.interpolate(x, size=(SIZE, SIZE), mode="bilinear", align_corners=False)
        loss = F.cross_entropy(model(x), torch.from_numpy(labels[idx]))
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        losses.append(loss.item())
    return model.eval(), losses

In [ ]:
STEPS_18 = 60 if FAST_MODE else 300     # the steps of the training that follows the diagnoses
with checked("B1.18"):
    ok_18 = diagnose_18(finetune_18)

In [ ]:
if not globals().get("ok_18"):
    print("⏳ Ex B1.18 : l'entraînement attend que les quatre points de la cellule précédente passent.")
else:
    with checked("B1.18"), torch.random.fork_rng():
        model_18, losses_18 = finetune_18(XT_B, YT_B, steps=STEPS_18)
        acc_18 = evaluate(model_18, X_TEST[nested(100, "test")], Y_TEST[nested(100, "test")])
        print(f"{STEPS_18} pas de 4 images : loss {fr(np.mean(losses_18[:10]), 2)} sur les 10 premiers pas, "
              f"{fr(np.mean(losses_18[-10:]), 2)} sur les 10 derniers ; accuracy de test {pct(acc_18)} (le hasard : 10 %).")
        verdict("B1.18", acc_18 > 0.3, "le fine-tuning réparé apprend.",
                "l'accuracy reste proche du hasard : une erreur silencieuse reste-t-elle ?")

### Ex B1.19 — Fine-tuning complet : warmup + cosinus, early stopping, meilleur checkpoint 📦 ★★★ ⏱️ 50 min 🚀
**Objectif :** écrire un fine-tuning complet avec learning rates par groupe, warmup puis cosinus pas à pas, early stopping et retour au meilleur checkpoint.  
**Prérequis :** Ex B1.17 · ✏️ B1.3 · ch. 23 (`fit()`, early stopping, meilleur checkpoint) · ch. 19 (warmup + cosinus) · fiche §201.4 · **Fil rouge :** CIFAR-10 · **Parcours :** R, C

**Contrat**
- `warmup_cosine_19(step, t_warmup, t_total)` : le facteur du learning rate au pas `step` (0, 1, 2…) : `step / t_warmup` pendant le warmup, puis un cosinus de 1 à 0 au pas `t_total`, et 0 au-delà ; 0 au pas 0, comme dans la fiche ; avec `t_warmup = 0`, pas de warmup ;
- `make_full_19(num_classes=10)` : ResNet-18 `IMAGENET1K_V1`, une nouvelle tête `nn.Linear(512, num_classes)`, **tous** les paramètres entraînables ; sur `DEVICE` ;
- `fine_tune_19(model, train, val, epochs, head_lr, gamma, t_warmup, patience, batch_size=32, seed=0)` (`train` et `val` sont des couples `(images, labels)`) :
  - un `AdamW` sur `param_groups_17` (B1.17) avec cinq modules, de la tête vers l'entrée : `model.fc`, `model.layer4`, `model.layer3`, `model.layer2`, puis `nn.ModuleList([model.conv1, model.bn1, model.layer1])` ;
  - un `LambdaLR` de facteur `warmup_cosine_19(t, t_warmup, t_total)`, avec `t_total = epochs × ceil(len(images) / batch_size)`, qui avance d'un pas après **chaque** `optimizer.step()` ;
  - chaque epoch : `model.train()`, `freeze_batchnorm_17(model)`, les mini-batches de `batches(X, y, batch_size, seed + epoch)` ; à la fin de l'epoch, **un** appel `evaluate(model, X_val, y_val)` ;
  - le meilleur checkpoint : une **copie** des poids de l'epoch à la meilleure accuracy de validation (une égalité ne compte pas comme un progrès) ; arrêt dès que `patience` epochs de suite n'ont pas battu ce record ; à la fin, ces poids sont rechargés dans `model`, laissé en mode `eval()` ;
  - rend `{"train_loss": [...], "val_acc": [...], "lr": [...], "best_epoch": ...}` : un `float` par epoch faite pour les deux premières, la liste des learning rates des groupes **utilisés** à chaque pas (lus avant `optimizer.step()`), et l'indice (à partir de 0) de la meilleure epoch.

La cellule de vérification contrôle ton planning sur quelques pas, puis lance ton fine-tuning sur 20 images avec une validation **scriptée** : `evaluate` rend tour à tour 0,30, 0,50, 0,40, 0,45, 0,60 et retient les poids qu'il voit, ce qui vérifie l'arrêt, `best_epoch`, les learning rates de chaque pas et les poids rendus. La cellule suivante fine-tune tout le réseau sur `XT_B` (`head_lr = 1e-3`, `gamma = 0,2`, une epoch de warmup ; ≈ 50 s en mode rapide), puis trace le planning de chaque groupe et la validation. 🚀 Mode complet : 10 000 images à 224 pixels, 10 epochs, une patience de 2 ; plusieurs heures sur CPU, quelques minutes par epoch sur un GPU de Colab.

Dans tes notes : compare l'accuracy de test à celles de B1.17 et du linear probe, sur les mêmes images. Pourquoi le meilleur checkpoint n'est-il pas forcément le dernier ? Pourquoi le planning avance-t-il à chaque mini-batch et pas à chaque epoch (✏️ B1.3) ? Que se passerait-il si l'on gardait `model.state_dict()` sans le copier ?

In [ ]:
def warmup_cosine_19(step, t_warmup, t_total):
    """The factor of the learning rate at step `step` (0, 1, 2...): step / t_warmup during the warmup, then a cosine
    from 1 down to 0 at step t_total (0 at step 0, as in the fiche)."""
    raise NotImplementedError("warmup_cosine_19")


def make_full_19(num_classes=10):
    """ResNet-18 IMAGENET1K_V1 with a new head nn.Linear(512, num_classes), every parameter trainable. On DEVICE."""
    raise NotImplementedError("make_full_19")


def fine_tune_19(model, train, val, epochs, head_lr, gamma, t_warmup, patience, batch_size=32, seed=0):
    """Full fine-tuning with warmup + cosine and early stopping; leaves in `model` the weights of its best epoch.

    Returns {"train_loss": [one float per epoch], "val_acc": [one float per epoch],
             "lr": [[the learning rate of each group] for each step], "best_epoch": index of the best epoch}."""
    raise NotImplementedError("fine_tune_19")

In [ ]:
with checked("B1.19"):
    for args_19, want_19 in (((0, 4, 20), 0.0), ((2, 4, 20), 0.5), ((4, 4, 20), 1.0), ((12, 4, 20), 0.5),
                             ((20, 4, 20), 0.0), ((25, 4, 20), 0.0), ((3, 0, 10), 0.5 * (1 + math.cos(math.pi * 0.3)))):
        got_19 = warmup_cosine_19(*args_19)
        if not returned("B1.19", "warmup_cosine_19", got_19):
            raise StopCheck
        if not math.isclose(float(got_19), want_19, abs_tol=1e-9):
            print(f"❌ Ex B1.19 : warmup_cosine_19{args_19} = {got_19!r}, attendu {want_19:.4f}.")
            raise StopCheck
    print("✅ Ex B1.19 : warmup_cosine_19 suit le warmup puis le cosinus (0 au pas 0, 1 à la fin du warmup, 0 à t_total).")

with checked("B1.19"):
    model_19 = make_full_19()
    if not returned("B1.19", "make_full_19", model_19):
        raise StopCheck
    verdict("B1.19", isinstance(getattr(model_19, "fc", None), nn.Linear) and model_19.fc.out_features == 10
            and all(p.requires_grad for p in model_19.parameters())
            and all(torch.equal(model_19.state_dict()[k].cpu(), v) for k, v in IMAGENET_RESNET18.items() if not k.startswith("fc.")),
            "make_full_19 : les poids IMAGENET1K_V1, une tête à 10 sorties, tout est entraînable.",
            "make_full_19 : attendu les poids IMAGENET1K_V1, model.fc = nn.Linear(512, 10) et tous les paramètres entraînables.")

with checked("B1.19"), torch.random.fork_rng():
    # a scripted validation: evaluate() returns these accuracies in turn and remembers the weights it saw
    script_19, seen_19 = [0.30, 0.50, 0.40, 0.45, 0.60], []
    evaluate_19 = evaluate

    def scripted_evaluate_19(model, X, y, batch_size=128):
        seen_19.append({k: v.detach().clone() for k, v in model.state_dict().items()})
        return script_19[len(seen_19) - 1] if len(seen_19) <= len(script_19) else 0.0

    tiny_19 = (X_TRAIN[nested(2)], Y_TRAIN[nested(2)])
    evaluate = scripted_evaluate_19
    try:
        hist_19 = fine_tune_19(make_full_19(), tiny_19, (X_VAL[:10], Y_VAL[:10]), epochs=5, head_lr=1e-3, gamma=0.5,
                               t_warmup=3, patience=2, batch_size=10, seed=0)
    finally:
        evaluate = evaluate_19
    if not returned("B1.19", "fine_tune_19", hist_19):
        raise StopCheck
    if not isinstance(hist_19, dict) or not {"train_loss", "val_acc", "lr", "best_epoch"} <= set(hist_19):
        print('❌ Ex B1.19 : attendu un dictionnaire {"train_loss", "val_acc", "lr", "best_epoch"}, reçu '
              f"{type(hist_19).__name__} {sorted(hist_19) if isinstance(hist_19, dict) else ''}.")
        raise StopCheck
    verdict("B1.19", len(seen_19) == 4 and list(hist_19["val_acc"]) == script_19[:4],
            "early stopping : avec une patience de 2, l'entraînement s'arrête après la 4ᵉ epoch (validation 0,30, 0,50, 0,40, 0,45).",
            f"avec les accuracies de validation 0,30, 0,50, 0,40, 0,45, 0,60 et une patience de 2, attendu 4 epochs "
            f"(et evaluate appelée une fois par epoch) ; reçu {len(seen_19)} appel(s) d'evaluate, val_acc = {hist_19['val_acc']!r:.80}.")
    verdict("B1.19", hist_19["best_epoch"] == 1, "best_epoch = 1 (l'epoch à 0,50).",
            f"best_epoch = {hist_19['best_epoch']!r}, attendu l'indice (à partir de 0) de l'epoch à 0,50.")
    lrs_19 = hist_19["lr"]
    n_steps_19 = 2 * min(len(seen_19), 5)           # 20 images in mini-batches of 10: 2 steps per epoch
    t_total_19 = 5 * 2                               # the planned total: epochs x steps per epoch
    lr_ok_19 = isinstance(lrs_19, list) and len(lrs_19) == n_steps_19 and all(
        sorted(round(v, 15) for v in row) == sorted(round(1e-3 * 0.5 ** k * warmup_cosine_19(t, 3, t_total_19), 15)
                                                     for k in range(5) for _ in range(2))
        for t, row in enumerate(lrs_19))
    verdict("B1.19", lr_ok_19, "learning rates : 5 modules × 2 groupes, head_lr × gamma^k × warmup_cosine_19(t), pas par pas "
            "(t_total = epochs × pas par epoch).",
            f"history['lr'] : attendu {n_steps_19} lignes (une par pas), chacune avec les 10 learning rates des groupes au pas t, "
            "lus avant optimizer.step() (relis le contrat : modules, gamma, t_warmup, t_total).")

with checked("B1.19"), torch.random.fork_rng():
    # the best checkpoint: the weights left in the model must be those seen at the best epoch, not the last ones
    seen_19 = []
    model_19 = make_full_19()
    evaluate = scripted_evaluate_19
    try:
        fine_tune_19(model_19, tiny_19, (X_VAL[:10], Y_VAL[:10]), epochs=5, head_lr=1e-3, gamma=0.5, t_warmup=3,
                     patience=2, batch_size=10, seed=0)
    finally:
        evaluate = evaluate_19
    if len(seen_19) >= 4:
        final_19 = model_19.state_dict()
        best_19 = all(torch.equal(final_19[k], seen_19[1][k]) for k in final_19)
        last_19 = all(torch.equal(final_19[k], seen_19[-1][k]) for k in final_19)
        verdict("B1.19", best_19, "le modèle rendu a les poids de la meilleure epoch (une copie, rechargée à la fin).",
                "le modèle rendu a les poids de la dernière epoch : as-tu gardé une COPIE des poids de la meilleure, et l'as-tu "
                "rechargée ?" if last_19 else "le modèle rendu n'a les poids d'aucune epoch évaluée : que recharges-tu à la fin ?")

In [ ]:
with checked("B1.19"), torch.random.fork_rng():
    torch.manual_seed(19)
    MODEL_19 = make_full_19()
    steps_per_epoch_19 = math.ceil(len(XT_B) / 32)
    start_19 = time.perf_counter()
    history_19 = fine_tune_19(MODEL_19, (XT_B, YT_B), (X_VAL, Y_VAL), epochs=FT_EPOCHS, head_lr=1e-3, gamma=0.2,
                              t_warmup=steps_per_epoch_19, patience=1 if FAST_MODE else 2)
    seconds_19 = time.perf_counter() - start_19
    test_19 = evaluate(MODEL_19, X_TEST, Y_TEST)
    print("validation par epoch :", ", ".join(pct(a) for a in history_19["val_acc"]),
          f"; meilleure epoch : {history_19['best_epoch']} ; test du meilleur modèle : {pct(test_19)} ({seconds_19:.0f} s)")
    lr_19 = np.array(history_19["lr"])
    fig_19, axes_19 = plt.subplots(1, 2, figsize=(10, 3.4))
    steps_19 = np.arange(1, len(lr_19))                 # step 0 has a learning rate of 0: no place on a log scale
    for k_19 in range(lr_19.shape[1]):
        axes_19[0].plot(steps_19, lr_19[1:, k_19], lw=1)
    axes_19[0].set(yscale="log", xlabel="pas", ylabel="learning rate (échelle log)",
                   title="Un planning par groupe (deux groupes par module)")
    axes_19[1].plot(range(len(history_19["val_acc"])), history_19["val_acc"], "o-")
    axes_19[1].axvline(history_19["best_epoch"], color="gray", ls="--", lw=1)
    axes_19[1].set_xticks(range(len(history_19["val_acc"])))
    axes_19[1].set(xlabel="epoch", ylabel="accuracy de validation", title="Validation et meilleur checkpoint")
    plt.tight_layout()
    plt.show()

## ✅ Bilan

**Auto-évaluation** (note-toi de 0 à 3 dans `mon_travail/suivi/auto_evaluation.md`) :
1. Sais-tu suivre les formes d'un mini-batch dans un ResNet pré-entraîné et compter ce que tu entraînes selon la stratégie ?
2. Sais-tu écrire les groupes de paramètres et le planning d'un fine-tuning, et dire pourquoi les premières couches bougent le moins ?
3. Sais-tu calculer les probabilités d'un zéro-shot de CLIP, et dire ce que fait l'échelle $s$ ?
4. Sais-tu dire ce qu'un modèle dérivé hérite de son modèle de base (licence, biais, limites) ?

**Pour aller plus loin** : le tutoriel de PyTorch sur le transfer learning et l'article de Yosinski et coll. (2014), cités dans la fiche. La suite : le bonus B2, tokenisation et embeddings.

*Les exercices B1.20 à B1.28 (parties C et D : quand le transfert aide ou échoue, modèles de fondation, coût et livraison) seront ajoutés à la prochaine étape de génération ; relance alors `python tools/start_chapter.py B1` pour obtenir le notebook complet.*